In [170]:
# Standard library
import os
import re
from pathlib import Path

# Third-party (alphabetical)
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer, AutoModel

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [171]:
import sys

cwd = Path.cwd()
proj = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(proj))

print("Added to sys.path:", sys.path[0])

Added to sys.path: /Users/pallavi_chandanshive/projects/clinical-summarization-eval


In [172]:
from src.llm.llm import generate_summary
from config.prompts import SYSTEM_PROMPT

In [173]:

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# Chunking configuration
FIXED_CHUNK_WORDS = 150
FIXED_CHUNK_OVERLAP = 30

# Columns we expect in the clinical notes dataset
REQUIRED_NOTE_COLUMNS = {
    "person_id",
    "admission_id",
    "clinical_note_id",
    "clean_note_text",
    "creation_timestamp",
    "note_subject",
    "note_type",
}

In [174]:
NOTES_PATH =  '../data/raw/clinical_notes.csv'   # update if filename/path differs

notes = pd.read_csv(NOTES_PATH)


print(f"Loaded {len(notes):,} clinical notes")
notes.head()

Loaded 1,602 clinical notes


,ingest_timestamp,clinical_note_id,clean_note_text,creation_timestamp,updt_dt_tm,note_subject,note_type,admission_id,person_id
0,07/01/2026 14:35,17bf845b-88f8-4604-8983-6e74453aada5,"Patient Name: Judith Ada Wells\n- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3\n- NHS Number: 272733208\n- Date of Birth: 15/05/84 (39 years old)\n- Gender: Female\n- Allergies: NKA\n- Current Medications: No current medications\n\nTriage Details:\n- Date:07/01/26\n- Time: 14:05\n- Triage Category: Category 2 (Urgent - potentially serious condition requiring prompt atention)\n- Chief Complaint: Severe headache after exertion, rated 8/10 in intensity\n- Nurse: Jasmine Freda Murray\n\nInitial Observations:\n- BP: 160/90 mmHg\n- HR: 88 bpm\n\nED Diagnosis:\n- RCVS\n\nNext Steps:\n- Decision to perform neurological assessment\n- Urgent investigations planned, including CT head\n- Admitting Consultant: Dr. Kevin Richard Martin\nNurse Jasmine Freda Murray \nNMC number: 20F4626L",07/01/2026 14:05,07/01/2026 14:35,ED Triage,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
1,07/01/2026 14:50,e5d9c0a4-299a-425e-abbc-27fabe9cb742,Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurological examination performed; no abnormalities detected. Decision made to proceed with CT head scan to rule out intracrranial causes for headache.\nNurse Chukwuebuka Okafor \nNMC number: 18D6896L,07/01/2026 14:20,07/01/2026 14:50,ED Triage Follow-Up,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
2,07/01/2026 15:15,1a711621-1094-4d0b-9cec-7925438e19cb,"- Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.\n - Date/Time: 07/01/26, 14:45.\n - Staff involved: Nurse Jasmine Freda Murray.\n - Chief Complaint: Severe headache after exertion, rated 8/10 in intensity.\n - Initial Observations: BP of 160/90 mmHg and HR of 88 bpm recorded during triage.\n - Event details: CT head scan performed to rule out intracranial causes for the headache. Findings: No evidence of intracranial haemorrhage or mass lesion.\n - Next steps: No immediate medication changes. Blood tests arranged for further investigation.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L",07/01/2026 14:45,07/01/2026 15:15,ED CT Head Scan Review,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
3,07/01/2026 15:45,bbbb3acb-d58e-414d-a0de-7c29553b5459,"Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current triage category: Category 2 (Urgent - potentially serious condition requiring prompt attention). Diagnosis: RCVS. Initial triage performed by Nurse Chukwuebuka Okafor at 14:20 recorded BP at 160/90 mmHg and HR at 88 bpm, with no abnormalities on a brief neurological examination. A CT head scan was performed at 14:45 by Nurse Jasmine Freda Murray, confirming no evidence of intracranial hemorrhage or mass lesion. At 15:15, blood samples were collected for FBC, renal panel, LFTs, and inflammatory markers. No immediate medication changes or additions were made at this time. Awaiting test results to guide further management plan. Care provided by Nurse Jasmine Freda M urray.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L",07/01/2026 15:15,07/01/2026 15:45,ED Investigations,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
4,07/01/2026 17:00,e3ec2bf0-baa8-4287-8698-592f70a4bccf,"Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nFemale\n\nNHS No.\n272733208\n\nDate/Time\n07/01/26 16:30\n\nSeen By\nDr. Victoria Ellen Holmes (with Dr. Rebecca Stephanie Norris, ED Consultant)\n\nPresenting Complaint\nSevere headache after exertion. Severe headache started suddenly after exertion earlier today. Patient describes it as the worst headache of her life. Pain is throbbing and locagted 

In [175]:
missing_columns = REQUIRED_NOTE_COLUMNS - set(notes.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

notes = notes.copy()

notes["creation_timestamp"] = pd.to_datetime(
    notes["creation_timestamp"],
    errors="coerce"
)

notes["clean_note_text"] = (
    notes["clean_note_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove rows without usable note text
notes = notes.loc[
    notes["clean_note_text"].ne("")
].reset_index(drop=True)

notes["word_count"] = (
    notes["clean_note_text"]
    .str.split()
    .str.len()
)

print(f"Notes:      {len(notes):,}")
print(f"Patients:   {notes['person_id'].nunique():,}")
print(f"Admissions: {notes['admission_id'].nunique():,}")

Notes:      1,602
Patients:   50
Admissions: 69


In [176]:
def create_whole_note_chunks(notes_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create one retrieval chunk per clinical note.
    """

    chunks = notes_df[
        [
            "person_id",
            "admission_id",
            "clinical_note_id",
            "creation_timestamp",
            "note_subject",
            "note_type",
            "clean_note_text",
        ]
    ].copy()

    chunks = chunks.rename(
        columns={"clean_note_text": "chunk_text"}
    )

    chunks["chunk_index"] = 0

    chunks["chunk_id"] = (
        chunks["clinical_note_id"].astype(str)
        + "_whole_0"
    )

    chunks["chunk_strategy"] = "whole_note"

    return chunks


whole_chunks = create_whole_note_chunks(notes)

print(f"Whole-note chunks: {len(whole_chunks):,}")
whole_chunks.head()

Whole-note chunks: 1,602


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_text,chunk_index,chunk_id,chunk_strategy
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,"Patient Name: Judith Ada Wells\n- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3\n- NHS Number: 272733208\n- Date of Birth: 15/05/84 (39 years old)\n- Gender: Female\n- Allergies: NKA\n- Current Medications: No current medications\n\nTriage Details:\n- Date:07/01/26\n- Time: 14:05\n- Triage Category: Category 2 (Urgent - potentially serious condition requiring prompt atention)\n- Chief Complaint: Severe headache after exertion, rated 8/10 in intensity\n- Nurse: Jasmine Freda Murray\n\nInitial Observations:\n- BP: 160/90 mmHg\n- HR: 88 bpm\n\nED Diagnosis:\n- RCVS\n\nNext Steps:\n- Decision to perform neurological assessment\n- Urgent investigations planned, including CT head\n- Admitting Consultant: Dr. Kevin Richard Martin\nNurse Jasmine Freda Murray \nNMC number: 20F4626L",0,17bf845b-88f8-4604-8983-6e74453aada5_whole_0,whole_note
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurological examination performed; no abnormalities detected. Decision made to proceed with CT head scan to rule out intracrranial causes for headache.\nNurse Chukwuebuka Okafor \nNMC number: 18D6896L,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_whole_0,whole_note
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,"- Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.\n - Date/Time: 07/01/26, 14:45.\n - Staff involved: Nurse Jasmine Freda Murray.\n - Chief Complaint: Severe headache after exertion, rated 8/10 in intensity.\n - Initial Observations: BP of 160/90 mmHg and HR of 88 bpm recorded during triage.\n - Event details: CT head scan performed to rule out intracranial causes for the headache. Findings: No evidence of intracranial haemorrhage or mass lesion.\n - Next steps: No immediate medication changes. Blood tests arranged for further investigation.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L",0,1a711621-1094-4d0b-9cec-7925438e19cb_whole_0,whole_note
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,"Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current triage category: Category 2 (Urgent - potentially serious condition requiring prompt attention). Diagnosis: RCVS. Initial triage performed by Nurse Chukwuebuka Okafor at 14:20 recorded BP at 160/90 mmHg and HR at 88 bpm, with no abnormalities on a brief neurological examination. A CT head scan was performed at 14:45 by Nurse Jasmine Freda Murray, confirming no evidence of intracranial hemorrhage or mass lesion. At 15:15, blood samples were collected for FBC, renal panel, LFTs, and inflammatory markers. No immediate medication changes or additions were made at this time. Awaiting test results to guide further management plan. Care provided by Nurse Jasmine Freda M urray.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L",0,bbbb3acb-d58e-414d-a0de-7c29553b5459_whole_0,whole_note
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,"Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nFemale\n\nNHS No.\n272733208\n\nDate/Time\n07/01/26 16:30\n\nSeen By\nDr. Victoria Ellen Holmes (with Dr. Rebecca Stephanie Norris

In [177]:
def split_fixed_words(
    text: str,
    chunk_size: int = 150,
    overlap: int = 30,
) -> list[str]:
    """
    Split text into fixed-size word chunks with overlap.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()

    if len(words) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]

        if not chunk_words:
            break

        chunks.append(" ".join(chunk_words))

        if end >= len(words):
            break

    return chunks


def create_fixed_word_chunks(
    notes_df: pd.DataFrame,
    chunk_size: int = 150,
    overlap: int = 30,
) -> pd.DataFrame:
    """
    Create fixed-size word chunks from each clinical note.
    """

    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_fixed_words(
            row.clean_note_text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_fixed_{chunk_index}"
                    ),
                    "chunk_strategy": "fixed_words",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)


fixed_chunks = create_fixed_word_chunks(
    notes,
    chunk_size=FIXED_CHUNK_WORDS,
    overlap=FIXED_CHUNK_OVERLAP,
)

print(f"Fixed-word chunks: {len(fixed_chunks):,}")
fixed_chunks.head()

Fixed-word chunks: 2,298


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_index,chunk_id,chunk_strategy,chunk_text
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,0,17bf845b-88f8-4604-8983-6e74453aada5_fixed_0,fixed_words,"Patient Name: Judith Ada Wells\n- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3\n- NHS Number: 272733208\n- Date of Birth: 15/05/84 (39 years old)\n- Gender: Female\n- Allergies: NKA\n- Current Medications: No current medications\n\nTriage Details:\n- Date:07/01/26\n- Time: 14:05\n- Triage Category: Category 2 (Urgent - potentially serious condition requiring prompt atention)\n- Chief Complaint: Severe headache after exertion, rated 8/10 in intensity\n- Nurse: Jasmine Freda Murray\n\nInitial Observations:\n- BP: 160/90 mmHg\n- HR: 88 bpm\n\nED Diagnosis:\n- RCVS\n\nNext Steps:\n- Decision to perform neurological assessment\n- Urgent investigations planned, including CT head\n- Admitting Consultant: Dr. Kevin Richard Martin\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_fixed_0,fixed_words,Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurological examination performed; no abnormalities detected. Decision made to proceed with CT head scan to rule out intracrranial causes for headache.\nNurse Chukwuebuka Okafor \nNMC number: 18D6896L
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,0,1a711621-1094-4d0b-9cec-7925438e19cb_fixed_0,fixed_words,"- Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.\n - Date/Time: 07/01/26, 14:45.\n - Staff involved: Nurse Jasmine Freda Murray.\n - Chief Complaint: Severe headache after exertion, rated 8/10 in intensity.\n - Initial Observations: BP of 160/90 mmHg and HR of 88 bpm recorded during triage.\n - Event details: CT head scan performed to rule out intracranial causes for the headache. Findings: No evidence of intracranial haemorrhage or mass lesion.\n - Next steps: No immediate medication changes. Blood tests arranged for further investigation.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,0,bbbb3acb-d58e-414d-a0de-7c29553b5459_fixed_0,fixed_words,"Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current triage category: Category 2 (Urgent - potentially serious condition requiring prompt attention). Diagnosis: RCVS. Initial triage performed by Nurse Chukwuebuka Okafor at 14:20 recorded BP at 160/90 mmHg and HR at 88 bpm, with no abnormalities on a brief neurological examination. A CT head scan was performed at 14:45 by Nurse Jasmine Freda Murray, confirming no evidence of intracranial hemorrhage or mass lesion. At 15:15, blood samples were collected for FBC, renal panel, LFTs, and inflammatory markers. No immediate medication changes or additions were made at this time. Awaiting test results to guide further management plan. Care provided by Nurse Jasmine Freda M urray.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_fixed_0,fixed_words,"Patient Judith Ada Wells Age 39 Sex Female NHS No. 272733208 Date/Time 07/01/26 16:30 Seen By Dr. Victoria Ellen

In [178]:
SECTION_HEADINGS = [
    "Presenting Complaint",
    "History of Presenting Illness",
    "History of Present Illness",
    "HPI",
    "Review of Systems",
    "Past Medical History",
    "PMH",
    "Medications",
    "Medication",
    "Allergies",
    "Social History",
    "Family History",
    "On Examination",
    "Examination",
    "Observations",
    "Investigations",
    "Test Results",
    "Results",
    "Assessment",
    "Impression",
    "Diagnosis",
    "Treatment",
    "Plan",
]

SECTION_PATTERN = re.compile(
    rf"(?im)^(?:{'|'.join(map(re.escape, SECTION_HEADINGS))})\s*:?\s*$"
)


def is_heading_only(text: str) -> bool:
    """
    Return True if the chunk contains only a recognized section heading
    and no clinical content.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) != 1:
        return False

    return bool(SECTION_PATTERN.fullmatch(lines[0]))


def split_by_sections(text: str) -> list[str]:
    """
    Split a clinical note using known section headings.

    - Keeps section heading together with its content.
    - Drops empty sections that contain only a heading.
    - Falls back to the whole note when usable section boundaries
      are not found.
    """

    matches = list(SECTION_PATTERN.finditer(text))

    if len(matches) < 2:
        return [text]

    chunks = []

    # Preserve text before first recognized heading
    prefix = text[:matches[0].start()].strip()

    if prefix:
        chunks.append(prefix)

    for index, match in enumerate(matches):

        start = match.start()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(text)

        section = text[start:end].strip()

        if section and not is_heading_only(section):
            chunks.append(section)

    return chunks


def create_section_chunks(
    notes_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create section-aware chunks from clinical notes.

    Notes without detectable sections remain whole.
    """

    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_by_sections(
            row.clean_note_text
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_section_{chunk_index}"
                    ),
                    "chunk_strategy": "section",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)


section_chunks = create_section_chunks(notes)

print(f"Section chunks: {len(section_chunks):,}")
section_chunks.head()

Section chunks: 5,189


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_index,chunk_id,chunk_strategy,chunk_text
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,0,17bf845b-88f8-4604-8983-6e74453aada5_section_0,section,"Patient Name: Judith Ada Wells\n- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3\n- NHS Number: 272733208\n- Date of Birth: 15/05/84 (39 years old)\n- Gender: Female\n- Allergies: NKA\n- Current Medications: No current medications\n\nTriage Details:\n- Date:07/01/26\n- Time: 14:05\n- Triage Category: Category 2 (Urgent - potentially serious condition requiring prompt atention)\n- Chief Complaint: Severe headache after exertion, rated 8/10 in intensity\n- Nurse: Jasmine Freda Murray\n\nInitial Observations:\n- BP: 160/90 mmHg\n- HR: 88 bpm\n\nED Diagnosis:\n- RCVS\n\nNext Steps:\n- Decision to perform neurological assessment\n- Urgent investigations planned, including CT head\n- Admitting Consultant: Dr. Kevin Richard Martin\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_section_0,section,Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurological examination performed; no abnormalities detected. Decision made to proceed with CT head scan to rule out intracrranial causes for headache.\nNurse Chukwuebuka Okafor \nNMC number: 18D6896L
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,0,1a711621-1094-4d0b-9cec-7925438e19cb_section_0,section,"- Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.\n - Date/Time: 07/01/26, 14:45.\n - Staff involved: Nurse Jasmine Freda Murray.\n - Chief Complaint: Severe headache after exertion, rated 8/10 in intensity.\n - Initial Observations: BP of 160/90 mmHg and HR of 88 bpm recorded during triage.\n - Event details: CT head scan performed to rule out intracranial causes for the headache. Findings: No evidence of intracranial haemorrhage or mass lesion.\n - Next steps: No immediate medication changes. Blood tests arranged for further investigation.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,0,bbbb3acb-d58e-414d-a0de-7c29553b5459_section_0,section,"Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current triage category: Category 2 (Urgent - potentially serious condition requiring prompt attention). Diagnosis: RCVS. Initial triage performed by Nurse Chukwuebuka Okafor at 14:20 recorded BP at 160/90 mmHg and HR at 88 bpm, with no abnormalities on a brief neurological examination. A CT head scan was performed at 14:45 by Nurse Jasmine Freda Murray, confirming no evidence of intracranial hemorrhage or mass lesion. At 15:15, blood samples were collected for FBC, renal panel, LFTs, and inflammatory markers. No immediate medication changes or additions were made at this time. Awaiting test results to guide further management plan. Care provided by Nurse Jasmine Freda M urray.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_section_0,section,"Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nFemale\n\nNHS No.\n272733208\n\nDate/Time\n07/01/26 16:30\n\nSeen By\nDr. Vic

In [179]:
chunk_summary = pd.DataFrame(
    {
        "strategy": [
            "Whole note",
            "Fixed words",
            "Section aware",
        ],
        "num_chunks": [
            len(whole_chunks),
            len(fixed_chunks),
            len(section_chunks),
        ],
        "avg_words_per_chunk": [
            whole_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            fixed_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            section_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),
        ],
    }
)

chunk_summary

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,1602,129.997503
1,Fixed words,2298,99.711053
2,Section aware,5189,40.124880


In [180]:
section_counts = (
    section_chunks
    .groupby("clinical_note_id")
    .size()
)

print(
    "Notes split into multiple sections:",
    (section_counts > 1).sum()
)

print(
    "Notes left whole:",
    (section_counts == 1).sum()
)

print(
    "Percentage split:",
    round(
        (section_counts > 1).mean() * 100,
        2,
    ),
    "%"
)

Notes split into multiple sections: 569
Notes left whole: 1033
Percentage split: 35.52 %


In [181]:
example_note_id = (
    notes
    .sort_values("word_count", ascending=False)
    .iloc[0]["clinical_note_id"]
)

example_note = notes.loc[
    notes["clinical_note_id"] == example_note_id,
    [
        "clinical_note_id",
        "note_subject",
        "word_count",
        "clean_note_text",
    ],
]

example_note


print("WHOLE NOTE")
print("=" * 80)

display(
    whole_chunks.loc[
        whole_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

print("\nFIXED WORD")
print("=" * 80)

display(
    fixed_chunks.loc[
        fixed_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

print("\nSECTION")
print("=" * 80)

display(
    section_chunks.loc[
        section_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

WHOLE NOTE


,chunk_index,chunk_text
93,0,"Clerking Doctor\nDr. Kelly Nicola Hayward (SpR)\n\nPresenting Complaint\nAcute confusion following minor fall\n\nHistory of Presenting Complaint\n- Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26)\n- No LOC reported but recalls feeling dazed afterward\n- Developed pprogressive confusion over the following hours\n- Unable to remember recent events clearly\n- Denies headache, visual changes, N&V, or limb weakness\n- No reported CP, palpitations, or SOB\n\nReview of Systems\n- CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures\n- CVS: Denies chest pain, palpitations, or syncope\n- Resp: No breathlessness or cough\n- GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit\n- Renal: No dysuria or haematuria\n- MSK: Reports mild right knee pain following the fall\n- Endo: No polyuria, polydipsia, or heat/cold intolerance\n- Other: No recent fever or weight changes\n\nPast Medical History\n- HTN\n- Mild osteoarthritis\n\nMedications\n- No regular medications\n- IV paracetamol 1g QID prescribed during ED stay to manage pain and prevent discomfort\n\nAllergies\nNone\n\nSocial History\n- Lives alone in a ground-floor flat\n- Independent with activities of daily living\n- Retired accountant\n- Non-smoker\n- Drinks alcohol occasionally, approximately 6 units per week\n- No recreational drug use\n\nFamily History\n- Father: Deceased, history of MI at age 67\n- Motherr: Deceased, history of HTN and stroke\n- No known family history of neurodegenerative or psychiatric disorders\n\nOn Examination\n- Alert but mildly confused, GCS 14/15 (disoriented to time)\n- Appears dehydrated with dry mucous membranes\n- No obvious signs of head trauma or external injuries\n- Neurological examination:\n - Cranial nerves: Intact, pupils equal and reactive to light bilaterally\n - Motor: Normal power (5/5) in all limbs\n - Sensory: No deficits detected in light touch, pinprick, or vibratkion sensation\n - Reflexes: Normal and symmetrical in all limbs, plantar reflexes downgoing bilaterally\n - Coordination: No dysmetria or intention tremor on finger-nose testing\n - Gait: Unsteady, requires assistance to walk; no gross abnormalities in stance or heel-to-toe walking\n- Cardiovascular: Heart sounds dual, no murmurs, no peripheral oedema\n- Respiratory: Equal air entry bilaterally, no added sounds\n- Abdomen: Soft, non-tender, no organomegaly\n- No meningeal signs\n\nObservations\nHR 88\nBP 142/86\nRR 16\nTemp 36.8\nSpO2 98% on room air\n\nInvestigations\n- CT head (02/01/26): DSubdural hygroma noted, no MLS\n- No further imaging performed or currently planned\n\nTest Results\n- FB: Normal\n- U&E: Mild hyponatremia (Na 132 mmol/L)\n- CRP: Normal\n- LFTs: Normal\n- Coagulation profile: Normal\n- Blood glucose: Normal\n- Pending: None\n\nImpression\nAcute confusion secondary to subdural hygroma and mild hypoNa, likely exacerbated by dehydration\n\nPlan\n- Commence IV 0.9% sodium chloride, 1L over 8 hours, to correct dehydration and mild hyponatremia\n- Prescribe IV paracetamol 1g QID to mnaage pain and prevent discomfort\n- Continue monitoring neurological status with GCS assessments every 4 hours\n- Repeat U&E in 24 hours to assess resposne to fluid resuscitation\n- Ensure adequate oral hydration once IV fluids are discontinued\n- Monitor for any progression of symptoms or development of focal neurological deficits\n- Liaise with neurology team for ongoing care and management\n- Document any further findings during the patientâ€™s stay\n\nAdmitting Consultant\nDr. Ismel Siddique (Neurology Consultant)\n\n\nDr. Kelly Nicola Hayward (Specialty Registrar) \nGMC number: 1746274"



FIXED WORD


,chunk_index,chunk_text
130,0,"Clerking Doctor Dr. Kelly Nicola Hayward (SpR) Presenting Complaint Acute confusion following minor fall History of Presenting Complaint - Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26) - No LOC reported but recalls feeling dazed afterward - Developed pprogressive confusion over the following hours - Unable to remember recent events clearly - Denies headache, visual changes, N&V, or limb weakness - No reported CP, palpitations, or SOB Review of Systems - CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures - CVS: Denies chest pain, palpitations, or syncope - Resp: No breathlessness or cough - GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit - Renal: No dysuria or haematuria - MSK: Reports mild right knee pain following the fall - Endo: No polyuria, polydipsia, or heat/cold intolerance - Other: No recent fever"
131,1,"habit - Renal: No dysuria or haematuria - MSK: Reports mild right knee pain following the fall - Endo: No polyuria, polydipsia, or heat/cold intolerance - Other: No recent fever or weight changes Past Medical History - HTN - Mild osteoarthritis Medications - No regular medications - IV paracetamol 1g QID prescribed during ED stay to manage pain and prevent discomfort Allergies None Social History - Lives alone in a ground-floor flat - Independent with activities of daily living - Retired accountant - Non-smoker - Drinks alcohol occasionally, approximately 6 units per week - No recreational drug use Family History - Father: Deceased, history of MI at age 67 - Motherr: Deceased, history of HTN and stroke - No known family history of neurodegenerative or psychiatric disorders On Examination - Alert but mildly confused, GCS 14/15 (disoriented to time) - Appears dehydrated with dry mucous membranes - No obvious signs"
132,2,"family history of neurodegenerative or psychiatric disorders On Examination - Alert but mildly confused, GCS 14/15 (disoriented to time) - Appears dehydrated with dry mucous membranes - No obvious signs of head trauma or external injuries - Neurological examination: - Cranial nerves: Intact, pupils equal and reactive to light bilaterally - Motor: Normal power (5/5) in all limbs - Sensory: No deficits detected in light touch, pinprick, or vibratkion sensation - Reflexes: Normal and symmetrical in all limbs, plantar reflexes downgoing bilaterally - Coordination: No dysmetria or intention tremor on finger-nose testing - Gait: Unsteady, requires assistance to walk; no gross abnormalities in stance or heel-to-toe walking - Cardiovascular: Heart sounds dual, no murmurs, no peripheral oedema - Respiratory: Equal air entry bilaterally, no added sounds - Abdomen: Soft, non-tender, no organomegaly - No meningeal signs Observations HR 88 BP 142/86 RR 16 Temp 36.8 SpO2 98% on room air"
133,3,"air entry bilaterally, no added sounds - Abdomen: Soft, non-tender, no organomegaly - No meningeal signs Observations HR 88 BP 142/86 RR 16 Temp 36.8 SpO2 98% on room air Investigations - CT head (02/01/26): DSubdural hygroma noted, no MLS - No further imaging performed or currently planned Test Results - FB: Normal - U&E: Mild hyponatremia (Na 132 mmol/L) - CRP: Normal - LFTs: Normal - Coagulation profile: Normal - Blood glucose: Normal - Pending: None Impression Acute confusion secondary to subdural hygroma and mild hypoNa, likely exacerbated by dehydration Plan - Commence IV 0.9% sodium chloride, 1L over 8 hours, to correct dehydration and mild hyponatremia - Prescribe IV paracetamol 1g QID to mnaage pain and prevent discomfort - Continue monitoring neurological status with GCS assessments every 4 hours - Repeat U&E in 24 hours to assess resposne to fluid resuscitation - Ensure adequate oral hydration once IV"
134,4,- Continue monitoring neurological status with GCS assessments every 4 hours - Repeat U&E in 24 hours to assess resposne to fluid resuscitation - Ensure a


SECTION


,chunk_index,chunk_text
336,0,Clerking Doctor\nDr. Kelly Nicola Hayward (SpR)
337,1,"Presenting Complaint\nAcute confusion following minor fall\n\nHistory of Presenting Complaint\n- Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26)\n- No LOC reported but recalls feeling dazed afterward\n- Developed pprogressive confusion over the following hours\n- Unable to remember recent events clearly\n- Denies headache, visual changes, N&V, or limb weakness\n- No reported CP, palpitations, or SOB"
338,2,"Review of Systems\n- CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures\n- CVS: Denies chest pain, palpitations, or syncope\n- Resp: No breathlessness or cough\n- GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit\n- Renal: No dysuria or haematuria\n- MSK: Reports mild right knee pain following the fall\n- Endo: No polyuria, polydipsia, or heat/cold intolerance\n- Other: No recent fever or weight changes"
339,3,Past Medical History\n- HTN\n- Mild osteoarthritis
340,4,Medications\n- No regular medications\n- IV paracetamol 1g QID prescribed during ED stay to manage pain and prevent discomfort
341,5,Allergies\nNone
342,6,"Social History\n- Lives alone in a ground-floor flat\n- Independent with activities of daily living\n- Retired accountant\n- Non-smoker\n- Drinks alcohol occasionally, approximately 6 units per week\n- No recreational drug use"
343,7,"Family History\n- Father: Deceased, history of MI at age 67\n- Motherr: Deceased, history of HTN and stroke\n- No known family history of neurodegenerative or psychiatric disorders"
344,8,"On Examination\n- Alert but mildly confused, GCS 14/15 (disoriented to time)\n- Appears dehydrated with dry mucous membranes\n- No obvious signs of head trauma or external injuries\n- Neurological examination:\n - Cranial nerves: Intact, pupils equal and reactive to light bilaterally\n - Motor: Normal power (5/5) in all limbs\n - Sensory: No deficits detected in light touch, pinprick, or vibratkion sensation\n - Reflexes: Normal and symmetrical in all limbs, plantar reflexes downgoing bilaterally\n - Coordination: No dysmetria or intention tremor on finger-nose testing\n - Gait: Unsteady, requires assistance to walk; no gross abnormalities in stance or heel-to-toe walking\n- Cardiovascular: Heart sounds dual, no murmurs, no peripheral oedema\n- Respiratory: Equal air entry bilaterally, no added sounds\n- Abdomen: Soft, non-tender, no organomegaly\n- No meningeal signs"
345,9,Observations\nHR 88\nBP 142/86\nRR 16\nTemp 36.8\nSpO2 98% on room air


In [182]:
chunk_summary

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,1602,129.997503
1,Fixed words,2298,99.711053
2,Section aware,5189,40.124880


In [184]:
section_counts = (
    section_chunks
    .groupby("clinical_note_id")
    .size()
)

print("Notes split into multiple sections:", (section_counts > 1).sum())
print("Notes left whole:", (section_counts == 1).sum())
print(
    "Percentage split:",
    round((section_counts > 1).mean() * 100, 2),
    "%"
)

Notes split into multiple sections: 569
Notes left whole: 1033
Percentage split: 35.52 %


In [185]:
section_chunks = section_chunks.copy()

section_chunks["chunk_word_count"] = (
    section_chunks["chunk_text"]
    .str.split()
    .str.len()
)

section_chunks["chunk_word_count"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

count    5189.000000
mean       40.124880
std        42.022617
min         1.000000
10%         7.000000
25%        11.000000
50%        29.000000
75%        52.000000
90%        88.000000
95%       130.000000
max       385.000000
Name: chunk_word_count, dtype: float64

In [186]:
for threshold in [5, 10, 20]:
    count = (section_chunks["chunk_word_count"] < threshold).sum()
    percentage = count / len(section_chunks) * 100

    print(
        f"Chunks < {threshold} words: "
        f"{count:,} ({percentage:.1f}%)"
    )

Chunks < 5 words: 321 (6.2%)
Chunks < 10 words: 1,078 (20.8%)
Chunks < 20 words: 2,190 (42.2%)


In [187]:
small_chunks = (
    section_chunks.loc[
        section_chunks["chunk_word_count"] < 10,
        [
            "clinical_note_id",
            "note_subject",
            "chunk_index",
            "chunk_word_count",
            "chunk_text",
        ],
    ]
    .sort_values("chunk_word_count")
)

print(f"Number of chunks <10 words: {len(small_chunks)}")

pd.set_option("display.max_colwidth", None)

small_chunks.head(30)

Number of chunks <10 words: 1078


,clinical_note_id,note_subject,chunk_index,chunk_word_count,chunk_text
4370,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,0,1,#NAME?
3246,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,0,1,#NAME?
812,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,0,1,#NAME?
4834,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,0,1,#NAME?
2526,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,0,1,#NAME?
1682,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,0,1,#NAME?
962,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,0,1,#NAME?
3012,6bc4db9a-6d69-460e-a8d8-1657f6c79dc9,ED Depart Summary,4,2,Allergies\nPollen
1071,13ee5295-939b-4ebf-add9-f8c971f66a42,PMWR Ortho Reg,2,2,Investigations\nNone
3242,10ffbd24-1f2e-4303-8daf-873967db1ea3,Neuro AMWR,2,2,Investigations\nNone


In [188]:
investigation_only = section_chunks.loc[
    section_chunks["chunk_text"].str.strip().eq("Investigations"),
    [
        "clinical_note_id",
        "note_subject",
        "chunk_index",
    ],
]

print(
    f"Standalone 'Investigations' chunks: "
    f"{len(investigation_only)}"
)

investigation_only.head(10)

Standalone 'Investigations' chunks: 0


,clinical_note_id,note_subject,chunk_index


In [190]:
small_chunk_text_counts = (
    section_chunks.loc[
        section_chunks["chunk_word_count"] < 10,
        "chunk_text"
    ]
    .str.strip()
    .value_counts()
    .head(30)
)

small_chunk_text_counts

chunk_text
Investigations\nNone                                                   62
Past Medical History\nNil                                              27
Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)                 26
Medications\nNil                                                       25
Medications\nNone                                                      21
Clinician Leading Ward Round\nDr. Marcus John Whitehead (SpR)          20
Allergies\nPollen                                                      19
Clinician Leading Ward Round\nDr. Tao Tang (SpR)                       18
Clinician Leading Ward Round\nDr. Stuart Thomas Payne (SpR)            16
Allergies\nNil                                                         14
Clinician Leading Ward Round\nDr. Brenda Veronica Miles (SpR)          14
Allergies\nNo known drug allergies                                     11
Clinician Leading Ward Round\nDr. Edward Aaron O'Connor (SpR)          10
Allergies\nNone            

In [191]:
section_chunks.loc[
    section_chunks["chunk_text"].str.contains(
        r"#NAME\?",
        na=False,
        regex=True,
    ),
    [
        "clinical_note_id",
        "note_subject",
        "chunk_text",
    ],
].head(20)

,clinical_note_id,note_subject,chunk_text
812,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,#NAME?
962,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,#NAME?
1682,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,#NAME?
2526,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,#NAME?
3246,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,#NAME?
4370,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,#NAME?
4834,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,#NAME?


In [192]:
name_error_ids = section_chunks.loc[
    section_chunks["chunk_text"].str.strip().eq("#NAME?"),
    "clinical_note_id"
].unique()

notes.loc[
    notes["clinical_note_id"].isin(name_error_ids),
    [
        "clinical_note_id",
        "note_subject",
        "clean_note_text",
    ]
]

,clinical_note_id,note_subject,clean_note_text
244,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,#NAME?
293,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,#NAME?
520,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,#NAME?
787,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,#NAME?
1014,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,#NAME?
1363,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,#NAME?
1498,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,#NAME?


In [193]:
INVALID_NOTE_VALUES = {"#NAME?"}

invalid_mask = (
    notes["clean_note_text"]
    .str.strip()
    .isin(INVALID_NOTE_VALUES)
)

print("Invalid notes removed:", invalid_mask.sum())

notes_clean = notes.loc[~invalid_mask].copy()

print("Notes before cleaning:", len(notes))
print("Notes after cleaning:", len(notes_clean))

Invalid notes removed: 7
Notes before cleaning: 1602
Notes after cleaning: 1595


In [194]:
whole_chunks = create_whole_note_chunks(notes_clean)

fixed_chunks = create_fixed_word_chunks(
    notes_clean,
    chunk_size=FIXED_CHUNK_WORDS,
    overlap=FIXED_CHUNK_OVERLAP,
)

section_chunks = create_section_chunks(notes_clean)

## Embedding Models

We compare three embedding approaches:

1. General-purpose: GeminiAI text-embedding-3-small
2. Retrieval-focused: BGE-M3
3. Biomedical retrieval: MedCPT

Each embedding model will be evaluated with the same three chunking strategies:
- Whole-note
- Fixed-size overlapping chunks
- Section-aware chunks

### 1. GeminiAI — text-embedding-3-small

In [127]:
%pip install -q openai


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [128]:
%pip install -q python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [195]:
from google import genai

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH, override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")

print("Gemini key loaded:", bool(gemini_api_key))

client = genai.Client(api_key=gemini_api_key)

Gemini key loaded: True


In [196]:
# Sanity test: generate one Gemini embedding

test_text = "Patient developed acute confusion following a minor fall."

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=test_text,
)

test_embedding = response.embeddings[0].values

print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding dimensions: 3072
First 10 values: [0.0114165805, -0.014645216, 0.005921604, -0.06154907, 0.00182117, -0.0041579907, -0.014994828, 0.0123460945, 0.0035066789, 0.001972333]


In [131]:
%pip install -q sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [132]:
%pip install nbqa ruff isort


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [197]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH, override=True)

hf_token = os.getenv("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

HF token loaded: True


In [198]:
from huggingface_hub import login

login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [199]:
print("HF token loaded:", bool(hf_token))

HF token loaded: True


In [200]:
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [201]:

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

print("ENV path:", ENV_PATH)
print("ENV exists:", ENV_PATH.exists())

load_dotenv(ENV_PATH, override=True)

hf_token = os.getenv("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

ENV path: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/.env
ENV exists: True
HF token loaded: True


In [202]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

/Users/pallavi_chandanshive/projects/clinical-summarization-eval


In [203]:
from sentence_transformers import SentenceTransformer

BGE_MODEL_PATH = PROJECT_ROOT / "models" / "bge-base-en-v1.5"

bge_model = SentenceTransformer(str(BGE_MODEL_PATH))

test_text = "Patient developed acute confusion following a minor fall."

bge_embedding = bge_model.encode(test_text)

print("Embedding dimensions:", len(bge_embedding))
print("First 10 values:", bge_embedding[:10])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimensions: 768
First 10 values: [-0.00121067 -0.01221776  0.00434872  0.00537568  0.02330717 -0.00891011
  0.05173944  0.01838303 -0.02286186 -0.01189964]


### 3. MedCPT — Biomedical retrieval embedding

In [140]:
%pip install -q transformers torch


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [204]:

MEDCPT_QUERY_PATH = PROJECT_ROOT / "models" / "MedCPT-Query-Encoder"
MEDCPT_ARTICLE_PATH = PROJECT_ROOT / "models" / "MedCPT-Article-Encoder"

print("Query model exists:", MEDCPT_QUERY_PATH.exists())
print("Article model exists:", MEDCPT_ARTICLE_PATH.exists())

Query model exists: True
Article model exists: True


In [205]:
query_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

query_model = AutoModel.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

article_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

article_model = AutoModel.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

query_model.eval()
article_model.eval()

print("MedCPT models loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT models loaded successfully


In [206]:
test_query = "What caused the patient's acute confusion?"

test_article = """
Patient developed acute confusion following a minor fall.
CT head showed a subdural hygroma. Mild hyponatremia and dehydration
were also documented.
"""

In [207]:
with torch.no_grad():

    query_inputs = query_tokenizer(
        test_query,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    query_output = query_model(**query_inputs)
    query_embedding = query_output.last_hidden_state[:, 0, :]


    article_inputs = article_tokenizer(
        test_article,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    article_output = article_model(**article_inputs)
    article_embedding = article_output.last_hidden_state[:, 0, :]

In [208]:
print("Query embedding shape:", query_embedding.shape)
print("Article embedding shape:", article_embedding.shape)

print("First 10 query values:")
print(query_embedding[0][:10])

print("\nFirst 10 article values:")
print(article_embedding[0][:10])

Query embedding shape: torch.Size([1, 768])
Article embedding shape: torch.Size([1, 768])
First 10 query values:
tensor([ 0.0749, -0.0908, -0.0609, -0.3313, -0.0904, -0.0067, -0.3357, -0.0689,
        -0.1430, -0.2947])

First 10 article values:
tensor([-0.3596, -0.0933, -0.0052, -0.3221, -0.1385, -0.1546, -0.6965, -0.4492,
        -0.0412,  0.0531])


In [209]:
%whos DataFrame

Variable                 Type         Data/Info
-----------------------------------------------
chunk_summary            DataFrame    Shape: (3, 3)
example_note             DataFrame    Shape: (1, 4)
fixed_chunks             DataFrame    Shape: (2291, 10)
investigation_only       DataFrame    Shape: (0, 3)
longitudinal_patients    DataFrame    Shape: (19, 4)
notes                    DataFrame    Shape: (1602, 10)
notes_clean              DataFrame    Shape: (1595, 10)
patient_fixed_chunks     DataFrame    Shape: (112, 10)
patient_section_chunks   DataFrame    Shape: (296, 10)
patient_summary          DataFrame    Shape: (50, 4)
patient_whole_chunks     DataFrame    Shape: (90, 10)
section_chunks           DataFrame    Shape: (5182, 10)
small_chunks             DataFrame    Shape: (1078, 5)
whole_chunks             DataFrame    Shape: (1595, 10)


In [210]:
def embed_gemini_texts(texts):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts
    )

    embeddings = [
        embedding.values
        for embedding in response.embeddings
    ]

    return np.array(embeddings)

In [211]:
sample_texts = fixed_chunks["chunk_text"].head(3).tolist()

In [215]:
sample_gemini_embeddings = embed_gemini_texts(sample_texts)

print("Number of embeddings:", len(sample_gemini_embeddings))
print("Embedding shape:", sample_gemini_embeddings.shape)
print("First embedding, first 10 values:")
print(sample_gemini_embeddings[0][:10])

Number of embeddings: 3
Embedding shape: (3, 3072)
First embedding, first 10 values:
[-0.0213872  -0.00441735 -0.02074947 -0.06816971 -0.00032535  0.02437343
  0.01302461  0.00564994 -0.01271442  0.03882335]


In [219]:
def embed_bge_texts(texts):
    # Generate normalized BGE embeddings for the input texts
    return bge_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

In [220]:
sample_texts = fixed_chunks["chunk_text"].head(3).tolist()

sample_bge_embeddings = embed_bge_texts(sample_texts)

print("Number of embeddings:", len(sample_bge_embeddings))
print("Embedding shape:", sample_bge_embeddings.shape)
print("First embedding, first 10 values:")
print(sample_bge_embeddings[0][:10])

Number of embeddings: 3
Embedding shape: (3, 768)
First embedding, first 10 values:
[-0.01965749  0.00617     0.01011455 -0.05260094 -0.01465944  0.02197611
  0.04083324  0.02032509  0.00994131 -0.01185225]


In [221]:
def embed_medcpt_articles(texts):
    inputs = article_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = article_model(**inputs)

    embeddings = outputs.last_hidden_state[:, 0, :]

    return embeddings.cpu().numpy()

In [222]:
sample_medcpt_embeddings = embed_medcpt_articles(sample_texts)

print("Number of embeddings:", len(sample_medcpt_embeddings))
print("Embedding shape:", sample_medcpt_embeddings.shape)
print("First embedding, first 10 values:")
print(sample_medcpt_embeddings[0][:10])

Number of embeddings: 3
Embedding shape: (3, 768)
First embedding, first 10 values:
[-0.12583002  0.03172001 -0.20081128 -0.31064138  0.15516433 -0.20821486
 -0.39055628 -0.27165523 -0.02699574 -0.1275537 ]


In [223]:
# ============================================================
# Configuration-selection experiment: choose one patient
# ============================================================
# We want a patient with multiple admissions and a sufficiently
# rich clinical history so that differences between chunking and
# embedding strategies have a chance to appear.
#
# This table does NOT select the patient automatically.
# It simply summarizes the available longitudinal patients so
# that we can make a deliberate choice.

patient_summary = (
    notes_clean
    .groupby("person_id")
    .agg(
        num_admissions=("admission_id", "nunique"),
        num_notes=("clinical_note_id", "nunique"),
        total_words=("word_count", "sum")
    )
    .reset_index()
)

# Keep only genuinely longitudinal patients:
# patients represented across more than one admission.
longitudinal_patients = (
    patient_summary[
        patient_summary["num_admissions"] > 1
    ]
    .sort_values(
        ["num_admissions", "num_notes", "total_words"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Number of longitudinal patients:", len(longitudinal_patients))

longitudinal_patients

Number of longitudinal patients: 19


,person_id,num_admissions,num_notes,total_words
0,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,2,90,10268
1,137b8481-4f1d-4b7f-babd-20f7117023ad,2,72,7982
2,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,2,70,11050
3,359014a1-10e6-4bd8-9ba7-513d021c971e,2,68,6272
4,04df53ea-55c1-48d9-84a1-1f15c133b29b,2,66,7250
5,ff8c4724-b7de-4189-bccf-cffddd4d6d44,2,66,6794
6,c50e236f-6b3d-41c8-9e16-7ec343cac820,2,60,7794
7,5e434d78-b2f6-4d88-b327-fff6ee50b901,2,54,6200
8,37b5ce4d-dcfd-4bb7-bee4-d597eb114703,2,52,7050
9,42149ae1-6a3c-471e-a002-cb7263e8bb8c,2,50,6684


In [ ]:
# ============================================================
# Inspect the strongest candidate patients
# ============================================================
# We show the patients with the richest longitudinal records.
# We are looking for a patient with:
#   - more than one admission
#   - enough notes to make retrieval meaningful
#   - enough clinical history for chunking differences to matter

longitudinal_patients.head(10)

,person_id,num_admissions,num_notes,total_words
0,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,2,90,10268
1,137b8481-4f1d-4b7f-babd-20f7117023ad,2,72,7982
2,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,2,70,11050
3,359014a1-10e6-4bd8-9ba7-513d021c971e,2,68,6272
4,04df53ea-55c1-48d9-84a1-1f15c133b29b,2,66,7250
5,ff8c4724-b7de-4189-bccf-cffddd4d6d44,2,66,6794
6,c50e236f-6b3d-41c8-9e16-7ec343cac820,2,60,7794
7,5e434d78-b2f6-4d88-b327-fff6ee50b901,2,54,6200
8,37b5ce4d-dcfd-4bb7-bee4-d597eb114703,2,52,7050
9,42149ae1-6a3c-471e-a002-cb7263e8bb8c,2,50,6684


In [224]:
# ============================================================
# Inspect the top 3 candidate longitudinal patients
# ============================================================
# Note count alone does not tell us whether a patient is a good
# configuration-selection case.
#
# We inspect the top candidates to check:
#   1. whether both admissions contain meaningful amounts of data
#   2. whether notes span different note types / clinical events
#   3. whether one admission completely dominates the record
#
# We will then select ONE patient and keep that patient fixed
# across all 9 chunking × embedding configurations.

top_candidate_ids = longitudinal_patients.head(3)["person_id"].tolist()

candidate_overview = (
    notes_clean[
        notes_clean["person_id"].isin(top_candidate_ids)
    ]
    .groupby(["person_id", "admission_id"])
    .agg(
        num_notes=("clinical_note_id", "nunique"),
        total_words=("word_count", "sum"),
        first_note=("creation_timestamp", "min"),
        last_note=("creation_timestamp", "max")
    )
    .reset_index()
    .sort_values(["person_id", "first_note"])
)

candidate_overview

,person_id,admission_id,num_notes,total_words,first_note,last_note
0,137b8481-4f1d-4b7f-babd-20f7117023ad,485363b3-6fcf-4fb9-b005-5aca9e90529a,36,3991,2026-06-01 11:25:00,2026-12-01 10:00:00
1,137b8481-4f1d-4b7f-babd-20f7117023ad,fcc73cbb-cbb3-45ea-a6af-7c428e724f64,36,3991,2026-06-01 11:25:00,2026-12-01 10:00:00
2,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,d8370160-e492-4a63-8751-df815f762726,35,5525,2026-05-01 00:05:00,2026-10-01 09:00:00
3,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,e3a9531a-de58-4142-b5fa-c6b8ccb39d1b,35,5525,2026-05-01 00:05:00,2026-10-01 09:00:00
4,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,7b266c9e-fb25-4209-b856-95bda6915f12,45,5134,2026-03-01 01:35:00,2026-10-01 10:15:00
5,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,8ab5fb09-1638-4054-a031-03291ddf8d07,45,5134,2026-03-01 01:35:00,2026-10-01 10:15:00


In [225]:
# ============================================================
# Check whether the two admission IDs actually contain
# different clinical notes for the same patient.
#
# The previous summary showed identical note counts, word counts,
# and date ranges for both admissions. Before selecting a patient
# for the experiment, we need to determine whether these are
# genuinely separate encounters or duplicated admission mappings.
# ============================================================

check_person_id = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

patient_check = (
    notes_clean[
        notes_clean["person_id"] == check_person_id
    ]
    [
        [
            "clinical_note_id",
            "admission_id",
            "creation_timestamp",
            "note_type",
            "note_subject"
        ]
    ]
    .sort_values(["creation_timestamp", "clinical_note_id"])
)

patient_check.head(20)

,clinical_note_id,admission_id,creation_timestamp,note_type,note_subject
1096,85834d83-cab8-440a-b4c4-c8f77e9bc658,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 01:35:00,ED,ED Triage Assessment
602,f191854f-167f-48e6-8311-84081c6fdb76,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 01:35:00,ED,ED Triage Assessment
1097,932bcfd6-3e9a-4b4a-b885-73b84cc2ac9f,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 02:15:00,ED,ED Oxygen Therapy Intervention
603,b43f1cce-ade5-40d6-aa30-f28d72389f1d,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 02:15:00,ED,ED Oxygen Therapy Intervention
1098,1fd2d3ad-f926-49d7-8d39-b5aec82a0f09,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 02:45:00,ED,ED Oxygen Therapy Update
604,7d564f2a-c0f7-41fe-9f2c-64c42c0a0d17,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 02:45:00,ED,ED Oxygen Therapy Update
1099,8bddd5d8-539b-4e59-bf20-05f99a9cff77,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 03:15:00,ED,ED Review
605,b838732f-1383-4aff-b5d8-49629a0edcea,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 03:15:00,ED,ED Review
606,1891df2e-6957-485e-a785-e0b93a925ecc,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary
1100,3910c077-88b6-48eb-b157-285365a86b91,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary


In [226]:
# Check whether the SAME clinical_note_id appears under
# more than one admission_id.

note_admission_counts = (
    patient_check
    .groupby("clinical_note_id")["admission_id"]
    .nunique()
)

print("Unique clinical notes:", patient_check["clinical_note_id"].nunique())

print(
    "Notes linked to more than one admission:",
    (note_admission_counts > 1).sum()
)

print(
    "Total rows:",
    len(patient_check)
)

Unique clinical notes: 90
Notes linked to more than one admission: 0
Total rows: 90


In [227]:
# ============================================================
# Check whether the patient's two admissions contain identical
# clinical TEXT or only follow the same synthetic note structure.
#
# The admissions have matching timestamps, note types and subjects.
# Before using this patient for the configuration-selection
# experiment, we need to know whether their clinical content
# actually differs.
# ============================================================

admission_ids = patient_check["admission_id"].unique()

admission_1 = (
    notes_clean[
        (notes_clean["person_id"] == check_person_id) &
        (notes_clean["admission_id"] == admission_ids[0])
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

admission_2 = (
    notes_clean[
        (notes_clean["person_id"] == check_person_id) &
        (notes_clean["admission_id"] == admission_ids[1])
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

# Compare the actual note text row-by-row.
text_matches = (
    admission_1["clean_note_text"].values
    == admission_2["clean_note_text"].values
)

print("Admission 1 notes:", len(admission_1))
print("Admission 2 notes:", len(admission_2))
print("Exactly identical note texts:", text_matches.sum())
print("Different note texts:", (~text_matches).sum())

Admission 1 notes: 45
Admission 2 notes: 45
Exactly identical note texts: 45
Different note texts: 0


In [228]:
# ============================================================
# Validate the apparent multi-admission patients
# ============================================================
# Some patients appear to have multiple admissions, but the patient
# inspected above had identical clinical-note text duplicated across
# two different admission IDs.
#
# Before selecting a patient for the configuration-selection
# experiment, check whether this duplication pattern occurs across
# ALL patients that appear to have more than one admission.
#
# For each patient, we compare the sets of note texts belonging to
# their admissions. If two admissions contain exactly the same set
# of note texts, they are flagged as identical.
# ============================================================

multi_admission_ids = (
    notes_clean.groupby("person_id")["admission_id"]
    .nunique()
)

multi_admission_ids = multi_admission_ids[
    multi_admission_ids > 1
].index

comparison_results = []

for person_id in multi_admission_ids:

    patient_notes = notes_clean[
        notes_clean["person_id"] == person_id
    ]

    admission_ids = patient_notes["admission_id"].unique()

    # This dataset currently appears to contain two admissions for
    # these patients. Compare their actual clinical text.
    if len(admission_ids) == 2:

        texts_1 = set(
            patient_notes[
                patient_notes["admission_id"] == admission_ids[0]
            ]["clean_note_text"]
        )

        texts_2 = set(
            patient_notes[
                patient_notes["admission_id"] == admission_ids[1]
            ]["clean_note_text"]
        )

        comparison_results.append({
            "person_id": person_id,
            "admission_1_notes": len(texts_1),
            "admission_2_notes": len(texts_2),
            "identical_text_sets": texts_1 == texts_2
        })

admission_duplication_check = pd.DataFrame(comparison_results)

print(
    "Patients checked:",
    len(admission_duplication_check)
)

print(
    "Patients with identical admission text:",
    admission_duplication_check["identical_text_sets"].sum()
)

print(
    "Patients with different admission text:",
    (~admission_duplication_check["identical_text_sets"]).sum()
)

admission_duplication_check

Patients checked: 19
Patients with identical admission text: 19
Patients with different admission text: 0


,person_id,admission_1_notes,admission_2_notes,identical_text_sets
0,04df53ea-55c1-48d9-84a1-1f15c133b29b,33,33,True
1,05192757-942f-460d-b4ff-004ec39cc5ee,23,23,True
2,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,19,19,True
3,137b8481-4f1d-4b7f-babd-20f7117023ad,36,36,True
4,1705dd0f-011a-492c-b006-b27e03f2f4ed,14,14,True
5,31f9612b-5a6b-48ea-887b-895772a83b99,22,22,True
6,359014a1-10e6-4bd8-9ba7-513d021c971e,34,34,True
7,37b5ce4d-dcfd-4bb7-bee4-d597eb114703,26,26,True
8,42149ae1-6a3c-471e-a002-cb7263e8bb8c,25,25,True
9,5e434d78-b2f6-4d88-b327-fff6ee50b901,27,27,True


In [229]:
# ============================================================
# Remove duplicated clinical notes across admission IDs
# ============================================================
# Data validation showed that all 19 patients with multiple
# admission IDs contain identical sets of clinical-note text
# across those admissions.
#
# These records have different clinical_note_id/admission_id
# values but duplicate clinical content. Keeping both copies
# would cause the summarization and retrieval experiments to
# process the same clinical evidence twice.
#
# We therefore create a NEW dataframe rather than modifying
# notes_clean, preserving the previous cleaning stage.
# ============================================================

notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Notes before cross-admission deduplication:", len(notes_clean))
print("Notes after cross-admission deduplication:", len(notes_dedup))
print("Duplicate records removed:", len(notes_clean) - len(notes_dedup))
print("Patients remaining:", notes_dedup["person_id"].nunique())

Notes before cross-admission deduplication: 1595
Notes after cross-admission deduplication: 1103
Duplicate records removed: 492
Patients remaining: 50


In [230]:
# ============================================================
# Check the chunking functions currently available
# ============================================================
# We have cleaned and deduplicated the source notes.
# Before regenerating the three experimental chunk datasets,
# inspect the functions already defined in this notebook so we
# reuse the exact same chunking implementation as before.

%whos function

Variable                   Type        Data/Info
------------------------------------------------
create_fixed_word_chunks   function    <function create_fixed_wo<...>rd_chunks at 0x126d97c10>
create_section_chunks      function    <function create_section_chunks at 0x127528300>
create_whole_note_chunks   function    <function create_whole_no<...>te_chunks at 0x126d7fb60>
embed_bge_texts            function    <function embed_bge_texts at 0x12cc0aa30>
embed_gemini_texts         function    <function embed_gemini_texts at 0x12cc0b690>
embed_medcpt_articles      function    <function embed_medcpt_articles at 0x12cc09fe0>
generate_summary           function    <function generate_summary at 0x126d622a0>
is_heading_only            function    <function is_heading_only at 0x12751cf60>
load_dotenv                function    <function load_dotenv at 0x119649010>
login                      function    <function login at 0x12729aae0>
prepare_for_generation     function    <function prepare_for_g

In [231]:
# ============================================================
# Regenerate all three chunking strategies after deduplication
# ============================================================
# IMPORTANT:
# The previous whole_chunks, fixed_chunks, and section_chunks
# were generated from notes_clean, which still contained 492
# duplicate clinical-note records.
#
# We now regenerate every chunking strategy from notes_dedup so
# that all later embedding/configuration experiments use only
# unique clinical evidence.
# ============================================================

whole_chunks = create_whole_note_chunks(notes_dedup)

fixed_chunks = create_fixed_word_chunks(notes_dedup)

section_chunks = create_section_chunks(notes_dedup)


# ------------------------------------------------------------
# Confirm the new chunk counts
# ------------------------------------------------------------

print("Unique source notes:", len(notes_dedup))
print("Whole-note chunks:", len(whole_chunks))
print("Fixed-size chunks:", len(fixed_chunks))
print("Section-aware chunks:", len(section_chunks))

Unique source notes: 1103
Whole-note chunks: 1103
Fixed-size chunks: 1589
Section-aware chunks: 3620


In [232]:
# ============================================================
# Select candidates for the 3 × 3 configuration experiment
# ============================================================
# After removing duplicate clinical content across admission IDs,
# admission count is no longer used to select the test patient.
#
# Instead, we rank patients by the richness of their UNIQUE
# chronological clinical record:
#   - number of unique notes
#   - total amount of clinical text
#   - time span covered by the notes
#
# We will inspect the strongest candidates and select ONE patient
# for all 9 chunking × embedding configurations.
# ============================================================

patient_candidates = (
    notes_dedup
    .groupby("person_id")
    .agg(
        num_notes=("clinical_note_id", "nunique"),
        total_words=("word_count", "sum"),
        first_note=("creation_timestamp", "min"),
        last_note=("creation_timestamp", "max")
    )
    .reset_index()
)

# Calculate how much chronological time each patient's record spans.
patient_candidates["span_days"] = (
    patient_candidates["last_note"] -
    patient_candidates["first_note"]
).dt.days

# Show patients with the richest unique records first.
patient_candidates = (
    patient_candidates
    .sort_values(
        ["num_notes", "total_words", "span_days"],
        ascending=False
    )
    .reset_index(drop=True)
)

patient_candidates.head(10)

,person_id,num_notes,total_words,first_note,last_note,span_days
0,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,45,5134,2026-03-01 01:35:00,2026-10-01 10:15:00,214
1,136c7916-4f9b-4e5c-bf01-77e9d2c681a2,37,4278,2026-02-01 21:15:00,2026-09-01 11:00:00,211
2,137b8481-4f1d-4b7f-babd-20f7117023ad,36,3991,2026-06-01 11:25:00,2026-12-01 10:00:00,182
3,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,35,5525,2026-05-01 00:05:00,2026-10-01 09:00:00,153
4,a9827c1c-fb54-4e5b-8bc6-d3099869e671,35,3926,2026-01-01 16:00:00,2026-08-01 10:00:00,211
5,6e93f9d9-213d-4f2c-a1f0-f475dacef554,34,3867,2026-02-01 09:00:00,2026-11-01 09:30:00,273
6,359014a1-10e6-4bd8-9ba7-513d021c971e,34,3136,2026-07-01 00:05:00,2026-12-01 09:00:00,153
7,04df53ea-55c1-48d9-84a1-1f15c133b29b,33,3625,2026-01-01 07:30:00,2026-08-01 09:00:00,212
8,ff8c4724-b7de-4189-bccf-cffddd4d6d44,33,3397,2026-01-01 12:00:00,2026-06-01 10:00:00,150
9,51f15281-8840-4fd0-92de-89188ab8d736,32,3611,2026-01-01 03:40:00,2026-05-01 16:00:00,120


In [233]:
# ============================================================
# Inspect the selected candidate patient's clinical record
# ============================================================
# Candidate was selected because it has the richest unique
# longitudinal record after deduplication:
#   - 45 unique clinical notes
#   - 5,134 total words
#   - 214-day chronological span
#
# Before locking this patient for the 3 × 3 configuration
# experiment, inspect the note sequence to confirm that the
# record contains meaningful clinical progression over time.
# ============================================================

SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

selected_patient_notes = (
    notes_dedup[
        notes_dedup["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Number of notes:", len(selected_patient_notes))
print("Total words:", selected_patient_notes["word_count"].sum())
print(
    "Date range:",
    selected_patient_notes["creation_timestamp"].min(),
    "to",
    selected_patient_notes["creation_timestamp"].max()
)

selected_patient_notes[
    [
        "creation_timestamp",
        "note_type",
        "note_subject",
        "word_count"
    ]
]

Number of notes: 45
Total words: 5134
Date range: 2026-03-01 01:35:00 to 2026-10-01 10:15:00


,creation_timestamp,note_type,note_subject,word_count
0,2026-03-01 01:35:00,ED,ED Triage Assessment,116
1,2026-03-01 02:15:00,ED,ED Oxygen Therapy Intervention,86
2,2026-03-01 02:45:00,ED,ED Oxygen Therapy Update,87
3,2026-03-01 03:15:00,ED,ED Review,94
4,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,396
5,2026-03-01 04:30:00,Respiratory Inpatients,Medical Clerking,262
6,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,169
7,2026-03-01 11:00:00,Physiotherapy Documentation,Diaphragmatic Breathing Exercises,68
8,2026-03-01 13:00:00,Respiratory Inpatients,GP update post-CTPA,28
9,2026-03-01 15:00:00,Physiotherapy Documentation,Mobility Aid and Pacing Education,118


In [234]:
SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

In [235]:
# ============================================================
# Prepare the selected patient's three chunk representations
# ============================================================
# The same patient's clinical record will be used for every
# configuration in the 3 × 3 experiment.
#
# The ONLY differences between configurations will be:
#   1. chunking strategy:
#        - whole-note
#        - fixed-size (150 words, 30-word overlap)
#        - section-aware
#
#   2. embedding approach:
#        - Gemini
#        - BGE-base-en-v1.5
#        - MedCPT
#
# Keeping the patient fixed allows us to compare the nine
# configurations on exactly the same underlying clinical record.
# ============================================================

patient_whole_chunks = (
    whole_chunks[
        whole_chunks["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

patient_fixed_chunks = (
    fixed_chunks[
        fixed_chunks["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

patient_section_chunks = (
    section_chunks[
        section_chunks["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)


# Confirm how many searchable chunks each strategy creates
# from exactly the same 45-note patient record.

print("Original unique notes:", len(selected_patient_notes))
print("Whole-note chunks:", len(patient_whole_chunks))
print("Fixed-size chunks:", len(patient_fixed_chunks))
print("Section-aware chunks:", len(patient_section_chunks))

Original unique notes: 45
Whole-note chunks: 45
Fixed-size chunks: 56
Section-aware chunks: 148


In [236]:
# ============================================================
# Single retrieval query for configuration selection
# ============================================================
# We use ONE summarization-oriented retrieval query for all nine
# chunking × embedding configurations.
#
# The query represents the information needed by the downstream
# longitudinal summarization task. Its wording remains identical
# across every configuration so that query formulation is held
# constant during the comparison.
# ============================================================

RETRIEVAL_QUERY = (
    "Retrieve the clinically relevant information needed to produce a "
    "comprehensive longitudinal summary of this patient's clinical history, "
    "including major diagnoses, treatments, investigations, clinical "
    "progression, and outcomes."
)

print(RETRIEVAL_QUERY)

Retrieve the clinically relevant information needed to produce a comprehensive longitudinal summary of this patient's clinical history, including major diagnoses, treatments, investigations, clinical progression, and outcomes.


In [237]:
# ============================================================
# Configuration 1: Whole-note chunks + Gemini embeddings
# ============================================================
# This is the first of our 9 chunking × embedding configurations.
#
# We embed:
#   1. the single fixed retrieval query
#   2. all 45 whole-note chunks for the selected patient
#
# Both query and chunks must be embedded using the SAME embedding
# model so that their vectors exist in the same embedding space.
#
# We are NOT generating the clinical summary yet.
# This step only creates the numerical representations needed
# to compare the query with the patient's whole-note chunks.
# ============================================================

# Embed the single fixed retrieval query.
gemini_query_embedding = embed_gemini_texts(
    [RETRIEVAL_QUERY]
)

# Embed all whole-note chunks belonging to the selected patient.
gemini_whole_embeddings = embed_gemini_texts(
    patient_whole_chunks["chunk_text"].tolist()
)

# Check that the expected number of embeddings was produced.
print("Query embedding shape:", gemini_query_embedding.shape)
print("Whole-note embedding shape:", gemini_whole_embeddings.shape)

Query embedding shape: (1, 3072)
Whole-note embedding shape: (45, 3072)


In [238]:
# ============================================================
# Configuration 1: Calculate query-to-chunk similarity
# Whole-note chunks + Gemini embeddings
# ============================================================
# We now compare the ONE retrieval-query embedding against
# each of the 45 whole-note embeddings.
#
# Cosine similarity measures how closely the direction of two
# vectors aligns in the embedding space.
#
# Higher similarity = the chunk is more semantically relevant
# to our longitudinal-summarization retrieval query.
#
# IMPORTANT:
# We are only RANKING the chunks here.
# We have NOT yet decided how many chunks will be retrieved.
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

# Compare:
#   1 query embedding
#       against
#   45 whole-note embeddings
#
# Result initially has shape (1, 45), so [0] converts it
# into a simple array containing one score per chunk.
gemini_whole_scores = cosine_similarity(
    gemini_query_embedding,
    gemini_whole_embeddings
)[0]

# Create a copy so the original patient chunk dataframe
# remains unchanged.
gemini_whole_results = patient_whole_chunks.copy()

# Attach each chunk's similarity score.
gemini_whole_results["similarity_score"] = gemini_whole_scores

# Rank chunks from most similar to least similar.
gemini_whole_ranked = (
    gemini_whole_results
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

print("Number of ranked chunks:", len(gemini_whole_ranked))

gemini_whole_ranked[
    [
        "creation_timestamp",
        "note_type",
        "note_subject",
        "similarity_score",
        "chunk_text"
    ]
].head(10)

Number of ranked chunks: 45


,creation_timestamp,note_type,note_subject,similarity_score,chunk_text
0,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075,"Patient: Abena Bonsu, 43 years old, admitted with progressive breathlessness due to chronic thromboembolic pulmonary HTN. Past medical history includes HTN and asthma. Allergic to nuts. Session conducted on 07/01/26 at 10:00 by Therapist Joan Winifred Shaw (Physical). \n\nThe session focused on light mobilisation and breathing exercises, including diaphragmatic breathing techniques to improve oxygenation and manage e xertional breathlessnes. The patient ambulated 10 metres using a walking frame with minimal assistance. Oxygen saturations remained above 92% on low-flow oxygen throughout the session. The patient tolerated the exercises well.\n\nPlan: Gradually increase mobilisation distance in future sessions. No changes to medication.\nTherapist Joan Winifred Shaw (Physical)"
1,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707,"Patient reviewed at bedside on 2026-01-06 at 15:00 by Therapist Joan Winifred Shaw. Phyiso session focused on breathing exercises and light mobility. Diaphragmatic breathing techniques were practised to enhance O2 exchange. Patient completed two short walks along the ward corridor with rest breaks as required. O2 sats remained above 92% throughout the session, and the patient tolerated the exercises well without significant SOB. Plan to progressively increase walking distance over the next two days to build endurance and assess readiness for discharge.\nTherapist Joan Winifred Shaw (Physical)"
2,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647,"Patient\nAbena Bonsu\n\nAge\n43\n\nSex\nFemale\n\nNHS No.:\n395107142\n\nDate/Time\n03/01/2604:00\n\nSeen By\nDr. Brenda Veronica Miles, ED Consultant\n\nPresenting Complaint\nProgressive breathlessness\n\nHistory of Presenting Complaint\n- Progressive worsening of SOB over the last 2 weks\n- SOB worse on exertion\n- No associated CP, haemoptysis, fever, or syncope reported\n- Describes a sensation of tightness in the chest but denies palpitations or wheezing\n\nPast Medical History\n- HTN\n- Asthma\n\nMedications\n- None reported by the patient at the time of admjssion\n\nAllergies\n- Nuts\n\nSocial History\n- Lives alone\n- Non-smoker\n- No alcohol consumption reported\n- Works as a school teacher\n\nFamily History\n- Father: HTN\n- Mother: Asthma\n\nSystems Review\n- Respiratory: Progressive exertional dyspnoea. No haemoptysis, wheezing, or significant cough.\n- Cardiovascular: No palpitations, chest pain, or syncope.\n- Gastrointestinal: No abdominal pain, nausea, or vomitimg.\n- Neurological: No dizziness, confusion, or focal neuro symptoms.\n- General: No fever or weight loss.\n\nOn Examination\n- Appears mildly distressed with laboured breathing\n- Cardiovascular: Elevated JVP. No peripheral oedema noted. Heart sounds normal, no murmurs.\n- Respiratory: Bilateral basal creps heard on aauscultation. No wheeze.\n- Abdomen: Mildly tenddr hepatomegaly. No ascites.\n- Peripheral vascular: No evidence of DVT. No calf tenderness or swelling.\n\nObservations\nBP 92/64 mmHg\nHR 112 bpm\nRR 28 breaths/min\nTemp 36.8Â°C\nSPO2 90% on 2L NC O2\n\nInvestigations\n- Chest X-ray: Bilateral pulmonary congestion, no focal consolidation or pneumothorax. - CT pulmonary angiogram and echocardiogram planed to confirm suspected chronic thromboembolic pulmonary HTN (pending results).\n\nTest Results\n- ABG: PaO2 8.5 kPa, PaCO2 3.8 kPa, pH 7.46, HCO3- 24 mmol/L (compensated rdspiratory alkalosis, mild hypoxia)\n- NT-proBNP: 4,500 pg/mL (elevated, subsequent result after earlier 3,200 pg/mL reading)\n- Blood tests: D-dimer elevated, other results pending.\n\nImpression\nSuspected chronic thromboembolic pulmonary HTN causing progressive breathlessness and cardiac strain.\n\nReferral\nAccepted by Respiratory HDU under Dr. Elizabeth Kathryn Singh. P

In [239]:
# ============================================================
# Compare candidate retrieval depths: k = 5, 10, 15
# ============================================================
# We already ranked all 45 whole-note chunks using:
#   - the fixed retrieval query
#   - Gemini embeddings
#   - cosine similarity
#
# Now we create three candidate retrieval sets.
# These will help us determine how much retrieved evidence is
# needed before we freeze k for the full 3 × 3 experiment.
# ============================================================

retrieved_top5 = gemini_whole_ranked.head(5).copy()
retrieved_top10 = gemini_whole_ranked.head(10).copy()
retrieved_top15 = gemini_whole_ranked.head(15).copy()

print("Top 5:", len(retrieved_top5))
print("Top 10:", len(retrieved_top10))
print("Top 15:", len(retrieved_top15))

# Show which clinical notes are added as retrieval depth increases.
for k, retrieved in [
    (5, retrieved_top5),
    (10, retrieved_top10),
    (15, retrieved_top15)
]:
    print(f"\n--- TOP {k} ---")

    display(
        retrieved[
            [
                "creation_timestamp",
                "note_type",
                "note_subject",
                "similarity_score"
            ]
        ]
    )

Top 5: 5
Top 10: 10
Top 15: 15

--- TOP 5 ---


,creation_timestamp,note_type,note_subject,similarity_score
0,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075
1,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707
2,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647
3,2026-10-01 09:30:00,Respiratory Medicine Inpatients,Discharge summary provided,0.644000
4,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,0.639754



--- TOP 10 ---


,creation_timestamp,note_type,note_subject,similarity_score
0,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075
1,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707
2,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647
3,2026-10-01 09:30:00,Respiratory Medicine Inpatients,Discharge summary provided,0.644000
4,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,0.639754
5,2026-03-01 03:15:00,ED,ED Review,0.639619
6,2026-08-01 16:00:00,Physiotherapy Documentation,Respiratory Physio: Group Therapy,0.634670
7,2026-05-01 19:00:00,Medicine Inpatients,Resp PMWR,0.631795
8,2026-08-01 09:00:00,Medicine Inpatients,Resp AMWR,0.630280
9,2026-07-01 17:30:00,Medicine Inpatients,Resp PMWR,0.630013



--- TOP 15 ---


,creation_timestamp,note_type,note_subject,similarity_score
0,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075
1,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707
2,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647
3,2026-10-01 09:30:00,Respiratory Medicine Inpatients,Discharge summary provided,0.644000
4,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,0.639754
5,2026-03-01 03:15:00,ED,ED Review,0.639619
6,2026-08-01 16:00:00,Physiotherapy Documentation,Respiratory Physio: Group Therapy,0.634670
7,2026-05-01 19:00:00,Medicine Inpatients,Resp PMWR,0.631795
8,2026-08-01 09:00:00,Medicine Inpatients,Resp AMWR,0.630280
9,2026-07-01 17:30:00,Medicine Inpatients,Resp PMWR,0.630013


In [241]:
# ============================================================
# Chronologically reorder the retrieved evidence for k = 5,10,15
# ============================================================
# Retrieval ranking decides WHICH notes are selected.
# Chronological sorting decides the ORDER in which the
# summarization LLM will read those selected notes.
#
# We keep everything else fixed:
# - same patient
# - same whole-note chunking
# - same Gemini embeddings
# - same retrieval query
# - same summarization model/prompt
#
# Only k will differ.
# ============================================================

top5_chronological = (
    retrieved_top5
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

top10_chronological = (
    retrieved_top10
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

top15_chronological = (
    retrieved_top15
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

for k, df in [
    (5, top5_chronological),
    (10, top10_chronological),
    (15, top15_chronological)
]:
    print(f"\n--- TOP {k} IN CHRONOLOGICAL ORDER ---")
    display(
        df[
            [
                "creation_timestamp",
                "note_type",
                "note_subject",
                "similarity_score"
            ]
        ]
    )


--- TOP 5 IN CHRONOLOGICAL ORDER ---


,creation_timestamp,note_type,note_subject,similarity_score
0,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647
1,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,0.639754
2,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707
3,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075
4,2026-10-01 09:30:00,Respiratory Medicine Inpatients,Discharge summary provided,0.644000



--- TOP 10 IN CHRONOLOGICAL ORDER ---


,creation_timestamp,note_type,note_subject,similarity_score
0,2026-03-01 03:15:00,ED,ED Review,0.639619
1,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647
2,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,0.639754
3,2026-05-01 19:00:00,Medicine Inpatients,Resp PMWR,0.631795
4,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707
5,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075
6,2026-07-01 17:30:00,Medicine Inpatients,Resp PMWR,0.630013
7,2026-08-01 09:00:00,Medicine Inpatients,Resp AMWR,0.630280
8,2026-08-01 16:00:00,Physiotherapy Documentation,Respiratory Physio: Group Therapy,0.634670
9,2026-10-01 09:30:00,Respiratory Medicine Inpatients,Discharge summary provided,0.644000



--- TOP 15 IN CHRONOLOGICAL ORDER ---


,creation_timestamp,note_type,note_subject,similarity_score
0,2026-03-01 01:35:00,ED,ED Triage Assessment,0.629335
1,2026-03-01 03:15:00,ED,ED Review,0.639619
2,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647
3,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,0.639754
4,2026-04-01 15:30:00,Occupational Therapy Documentation,Energy Conservation Strategies,0.630006
5,2026-05-01 19:00:00,Medicine Inpatients,Resp PMWR,0.631795
6,2026-06-01 09:00:00,Medicine Inpatients,Resp AMWR,0.629943
7,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707
8,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075
9,2026-07-01 17:30:00,Medicine Inpatients,Resp PMWR,0.630013


In [251]:
# ============================================================
# Build chronological RAG context for k = 5, 10, and 15
# ============================================================
# Each dataframe is already in chronological order.
# Here we combine its retrieved chunks into one text context
# that will be passed to the summarization LLM.
#
# IMPORTANT:
# The clinical content is unchanged.
# Only the number of retrieved chunks differs between the
# three candidate retrieval depths.
# ============================================================

def build_rag_context(retrieved_df):
    context_parts = []

    for _, row in retrieved_df.iterrows():

        # Keep timestamp and note metadata so the LLM can
        # correctly understand the chronology of the evidence.
        note = (
            f"Date/Time: {row['creation_timestamp']}\n"
            f"Note Type: {row['note_type']}\n"
            f"Note Subject: {row['note_subject']}\n"
            f"{row['chunk_text']}"
        )

        context_parts.append(note)

    # Clearly separate individual clinical notes.
    return "\n\n---\n\n".join(context_parts)


context_k5 = build_rag_context(top5_chronological)
context_k10 = build_rag_context(top10_chronological)
context_k15 = build_rag_context(top15_chronological)


# Quick check that all three contexts were created.
print("k=5 context words:", len(context_k5.split()))
print("k=10 context words:", len(context_k10.split()))
print("k=15 context words:", len(context_k15.split()))

k=5 context words: 877
k=10 context words: 1565
k=15 context words: 2350


In [250]:
# ============================================================
# Prepare retrieved RAG evidence for the existing generate_summary()
# ============================================================
# generate_summary() expects:
#   - person_id
#   - clean_note_text
#
# Our retrieved evidence currently stores the text in chunk_text,
# so we create temporary dataframes in the format the function expects.
# ============================================================

def prepare_for_generation(retrieved_df, patient_id):
    generation_df = retrieved_df.copy()

    # The existing generation function expects this column name.
    generation_df["clean_note_text"] = generation_df["chunk_text"]

    # Ensure all rows belong to the selected patient.
    generation_df["person_id"] = patient_id

    return generation_df


k5_generation_df = prepare_for_generation(
    top5_chronological,
    SELECTED_PERSON_ID
)

k10_generation_df = prepare_for_generation(
    top10_chronological,
    SELECTED_PERSON_ID
)

k15_generation_df = prepare_for_generation(
    top15_chronological,
    SELECTED_PERSON_ID
)

In [249]:

groq_key = os.getenv("GROQ_API_KEY")

print("Groq key loaded:", groq_key is not None)
print("Key prefix:", groq_key[:4] if groq_key else "NO KEY")

Groq key loaded: True
Key prefix: gsk_


In [252]:
# List the models available to your current Groq API key/project
models = client.models.list()

for model in models.data:
    print(model.id)

qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
groq/compound-mini
whisper-large-v3-turbo
allam-2-7b
groq/compound
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b


In [248]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

model = client.models.retrieve("openai/gpt-oss-120b")
print(model)

Model(id='openai/gpt-oss-120b', created=1754408224, object='model', owned_by='OpenAI', active=True, context_window=131072, public_apps=None, max_completion_tokens=65536, hugging_face_id='openai/gpt-oss-120b', name='GPT OSS 120B', input_modalities=['text'], output_modalities=['text'], context_length=131072, max_output_length=65536, pricing={'prompt': '0.00000015', 'completion': '0.0000006', 'image': '0', 'request': '0', 'input_cache_read': '0.000000075'}, supported_sampling_parameters=['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], supported_features=['tools', 'json_mode', 'structured_outputs', 'reasoning'])


In [253]:
summary_k5 = generate_summary(
    SELECTED_PERSON_ID,
    k5_generation_df,
    SYSTEM_PROMPT
)

summary_k10 = generate_summary(
    SELECTED_PERSON_ID,
    k10_generation_df,
    SYSTEM_PROMPT
)

summary_k15 = generate_summary(
    SELECTED_PERSON_ID,
    k15_generation_df,
    SYSTEM_PROMPT
)

print("\n===== SUMMARY k=5 =====\n")
print(summary_k5)

print("\n===== SUMMARY k=10 =====\n")
print(summary_k10)

print("\n===== SUMMARY k=15 =====\n")
print(summary_k15)


===== SUMMARY k=5 =====

**Chronological Clinical Summary – Abena Bonsu, 43‑year‑old female**

**01 Jan 2026 – Emergency Department (04:00)**  
- **Presentation:** 2‑week history of progressive exertional dyspnoea; chest tightness without chest pain, haemoptysis, fever or syncope.  
- **Past history:** Hypertension, asthma.  
- **Allergies:** Nuts.  
- **Examination:** Mild distress, laboured breathing, elevated JVP, basal bilateral crepitations, mild tender hepatomegaly; no peripheral oedema, wheeze, or DVT signs.  
- **Vitals:** BP 92/64 mmHg, HR 112 bpm, RR 28 /min, Temp 36.8 °C, SpO₂ 90 % on 2 L nasal cannula.  
- **Investigations:**  
  - CXR – bilateral pulmonary congestion, no consolidation or pneumothorax.  
  - ABG – PaO₂ 8.5 kPa, PaCO₂ 3.8 kPa, pH 7.46, HCO₃⁻ 24 mmol/L (compensated respiratory alkalosis, mild hypoxia).  
  - NT‑proBNP 4 500 pg/mL (elevated; prior reading 3 200 pg/mL).  
  - D‑dimer – elevated (other labs pending).  
- **Impression:** Suspected chronic thromb

In [254]:
# Select the top 20 and top 25 retrieved notes
retrieved_top20 = gemini_whole_ranked.head(20).copy()
retrieved_top25 = gemini_whole_ranked.head(25).copy()

# Reorder selected notes chronologically before summarization
top20_chronological = (
    retrieved_top20
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

top25_chronological = (
    retrieved_top25
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

In [255]:
# Convert retrieved chunks into the format expected by generate_summary()
k20_generation_df = prepare_for_generation(
    top20_chronological,
    SELECTED_PERSON_ID
)

k25_generation_df = prepare_for_generation(
    top25_chronological,
    SELECTED_PERSON_ID
)

In [256]:
summary_k20 = generate_summary(
    SELECTED_PERSON_ID,
    k20_generation_df,
    SYSTEM_PROMPT
)

summary_k25 = generate_summary(
    SELECTED_PERSON_ID,
    k25_generation_df,
    SYSTEM_PROMPT
)

print("\n===== SUMMARY k=20 =====\n")
print(summary_k20)

print("\n===== SUMMARY k=25 =====\n")
print(summary_k25)


===== SUMMARY k=20 =====

**Longitudinal Clinical Summary – Abena Bonsu, 43‑year‑old female**

**03 Jan 2026 – Presentation & Initial Management**  
- Arrived to ED at 01:35 with progressive breathlessness (2‑week history, worse on exertion).  
- Past medical history: hypertension, asthma. Allergy: nuts. No regular medications reported.  
- Vital signs: SpO₂ 89 % on room air, BP 92/64 mmHg, HR 112 bpm, RR 28 /min, Temp 36.8 °C.  
- ED diagnosis: chronic thromboembolic pulmonary hypertension (CTEPH).  
- Immediate treatment: low‑flow oxygen via nasal cannula (improved SpO₂ to 92 %).  
- Investigations ordered: chest X‑ray (showed bilateral pulmonary congestion), arterial blood gas (PaO₂ 8.5 kPa, compensated respiratory alkalosis), NT‑proBNP 3 200 pg/mL (later 4 500 pg/mL), D‑dimer elevated.  

**04 Jan 2026 – Admission to Respiratory HDU**  
- Transferred under Dr Elizabeth K. Singh.  
- Anticoagulation started with therapeutic enoxaparin 1.5 mg/kg daily.  
- Planned CT pulmonary angio

In [257]:
# Build the complete chronological source record from all 45 unique notes.
# This will act as the reference against which each k-summary is evaluated.

reference_notes = (
    selected_patient_notes
    .sort_values("creation_timestamp")
    .copy()
)

reference_text = "\n\n---\n\n".join(
    reference_notes["clean_note_text"].astype(str).tolist()
)

print("Number of source notes:", len(reference_notes))
print("Reference word count:", len(reference_text.split()))
print("Reference character count:", len(reference_text))

Number of source notes: 45
Reference word count: 5178
Reference character count: 36069


In [258]:
# Prompt used only for selecting retrieval depth (k).
# The evaluator must judge the candidate summary strictly against
# the complete 45-note source record.

K_SELECTION_EVAL_PROMPT = """
You are evaluating a generated longitudinal clinical summary against
the complete source clinical record for the same patient.

Use ONLY the provided source clinical notes as the reference.

Evaluate the candidate summary for:

1. MAJOR CLINICAL COVERAGE
   Identify clinically important diagnoses, investigations, treatments,
   changes in clinical status, and outcomes present in the source record
   but missing from the candidate summary.

2. UNSUPPORTED CLAIMS
   Identify claims in the candidate summary that are not supported by
   the source clinical notes.

3. CONTRADICTIONS OR FACTUAL ERRORS
   Identify incorrect diagnoses, medications, doses, investigation
   results, clinical findings, numerical values, or other factual errors.

4. TEMPORAL ACCURACY
   Identify events that are assigned an incorrect date, placed in the
   wrong chronological order, or otherwise misrepresent the temporal
   clinical course.

5. OVERALL ASSESSMENT
   Assess whether the candidate summary adequately represents the major
   longitudinal clinical trajectory contained in the complete source record.

Do not use outside medical knowledge to add or infer facts.
Do not penalize the summary merely for omitting minor or repetitive details.
Focus on clinically meaningful information.

Return your evaluation in the following format:

MAJOR OMISSIONS:
- ...

UNSUPPORTED CLAIMS:
- ...

CONTRADICTIONS / FACTUAL ERRORS:
- ...

TEMPORAL ERRORS:
- ...

OVERALL ASSESSMENT:
...
"""

In [270]:
# Embed the same retrieval query using BGE.
bge_query_embedding = embed_bge_texts([RETRIEVAL_QUERY])

# Embed all 45 whole-note chunks for the selected patient using BGE.
bge_whole_embeddings = embed_bge_texts(
    patient_whole_chunks["chunk_text"].tolist()
)

print("Query embedding shape:", bge_query_embedding.shape)
print("Whole-note embeddings shape:", bge_whole_embeddings.shape)

Query embedding shape: (1, 768)
Whole-note embeddings shape: (45, 768)


In [271]:
from sklearn.metrics.pairwise import cosine_similarity

# Compare the retrieval query with each of the 45 whole-note chunks.
# This gives one similarity score for every chunk.
bge_whole_scores = cosine_similarity(
    bge_query_embedding,
    bge_whole_embeddings
)[0]

# Keep the chunk information and attach its BGE similarity score.
bge_whole_results = patient_whole_chunks.copy()
bge_whole_results["similarity_score"] = bge_whole_scores

# Rank chunks from most semantically relevant to least relevant.
bge_whole_ranked = (
    bge_whole_results
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

# Select the same fixed retrieval depth we chose earlier.
bge_whole_top20 = bge_whole_ranked.head(20).copy()

print("Total ranked chunks:", len(bge_whole_ranked))
print("Retrieved chunks:", len(bge_whole_top20))

# Look at the ranking before we move to summary generation.
bge_whole_top20[
    ["creation_timestamp", "similarity_score", "chunk_text"]
].head(20)

Total ranked chunks: 45
Retrieved chunks: 20


,creation_timestamp,similarity_score,chunk_text
0,2026-07-01 17:30:00,0.574053,"Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness\n\nIssues\n1. Chronic thromboembolic pulmonary hypertension (CTEPH)\n-Recent V/Q scan confirms mismatched perfusion defects\n-Stable oxygen saturation at 95% on room air\n-Initiated long-term anticoagulation therapy with Rivaroxaban 20 mg OD\n\nToday\n\nOn Review\nFeeling less SOB. Mobilising lightly.\n\nInvestigations\nRecent V/Q scan confirms mismatched perfusion defects. No new blood test results reported. Pending referral feedback from tertiary pulmonary HTN centre.\n\nObservations\nHR: 88\nBP: 112/72\nRR: 18\nTemp: 36.8\nSpO2: 95% RA\n\nOn Examination\nAlert, not in distress. No peripheral oedema. JVP not raised. Chest clear bilaterally. No murmurs or added heart sounds. Abdomen soft, non-tender. No hepatomegaly.\n\nPlan\n1. Start Rivaroxaban 20 mg OD for long-term anticoagulation. 2. Ensure pharmacy provides medicatiob pre-discharge. 3. Continue sildenafil 20 mg TID. 4. Complete discharge planning and confirm tertiary referral to PH centre. 5. Arrange OP follow-up and home O2 assessment. 6. Await feedback from tertiary PH centre regarding referral.\n\n\nDr. Sade Olowoyeye (Specialty Registrar) \nGMC number: 1359799"
1,2026-03-01 17:30:00,0.566744,"Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness\n\nIssues\n1. Chronic thromboembolic pulmonary hypertension (CTEPH)\n-CTPA confirmed chronic thromboembolivc disease\n-NT-proBNP significantly elevated (4,500 pg/mL)\n-Awaiting echo results for RV function\n\nToday\n\nOn Review\nReports no new symptoms. Comfortable on low-flow O2.\n\nInvestigations\nCYPA confirmed chronic thromboembolic disease. Awaiting echo results for RV function.\n\nObservations\nHR: 96\nBP: 118/72\nRR: 20\nTemp: 36.8\nSpO2: 94% on 2L O2\n\nOn Examination\nAlert, NAD. JVP raised. Heart sounds normal, no murmurs. Bibasal creps. No peripheral oedema.\n\nPlan\n1. Await echo results to assess RV function and determine se verity of pulm HTN. \n2. Note: Anticoagulation with apixaban 10 mg BD already initiated earlier today. Plan to optimise dose based on findings. \n3. Diaphragmatic breathing exercises to continue as instructed by physio. \n4. Physio review to continue for mobility aids and respiratory support.\n\n\nDr. Sade Olowoyeye (Specialty Registrar) \nGMC number: 1359799"
2,2026-09-01 17:00:00,0.559272,"Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness\n\nIssues\n1. Chronic Thromboembolic Pulmonary Hypertension (CTEPH)\n-Confirmed by V/Q scan showing mismatched perfusion defects\n-Persistently raised NT-proBNP\n-Improved oxygenation on room air\n\nToday\n\nOn Review\nReports reduced SOB. Mobilising well. No new sx.\n\nInvestigations\nV/Q scan earlier confirmed mismatched perfusion defects. Mild anaemia (Hb 110 g/L), NT-proBNP persistently raised.\n\nObservations\nHR: 88\nBP: 122/78\nRR: 16\nTemp: 36.8\nSpO2: 94% on room air\n\nOn Examination\nAlert, NAD. Chest clear. JVP not elevated. No peripheral oedema. Mobilising independently.\n\nPlan\n1.Finalised discharge medications: Rivaroxaban 15 mg BD for 21 days, then 20 mg OD. 2. Follow-up at tertiary PH centre in 4 weeks. 3. Review Hb and NT-proBNP in OPD.\n\n\nDr. Sade Olowoyeye (Specialty Registrar) \nGMC number: 1359799"
3,2026-10-01 08:00:00,0.559239,"Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness\n\nIssues\n1. Chronic Thromboembolic Pulmonary Hypertension (CTEPH)\n-Persistently elevated NT-proBNP\n-Mismatched perfusion defects on V/Q scan\n-Right ventricular strain on echocardiogram\n2. Mild anemia\n-Hb 110 g/L\n-Likely secondary to chronic disease\n\nToday\n\nOn Review\nFeelswell. Reduced SOB. Mobilising well.\n\nInvestigations\nHb 110 g/L, NT-proBNP persistently elevated. No new abnormalities.\n\nOb

In [272]:
# Keep exactly the 20 chunks selected by BGE,
# but reorder them by time before longitudinal summarization.
bge_whole_top20_chrono = (
    bge_whole_top20
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Retrieved chunks:", len(bge_whole_top20_chrono))

bge_whole_top20_chrono[
    ["creation_timestamp", "similarity_score", "chunk_text"]
]

Retrieved chunks: 20


,creation_timestamp,similarity_score,chunk_text
0,2026-03-01 03:15:00,0.535651,"- Date: 03/01/26\n - Time: 03:15\n - Staff: Nurse Audrey Phyllis Gill\n - Chief complaint: Progressive breathlessness.\n - Past medical history: HTN, Asthma.\n - Initial management: Patient placed on low-flow oxygen therapy to improve oxygenation, with SpO2 improving from 89%to 92%.\n - ABG results reviewed: PaO2 8.5 kPa, indicating mild hypoxia; compensated respiratory alkalosis likely due to hyperventilation.\n - NT-proBNP elevated at 3,200 pg/mL, suggesting significant cardiac strain.\n - Findings escalated to the medical team for further evaluation and consideration of imaging to investigate suspected pulmonary HTN.\nNurse Audrey Phyllis Gill \nNMC number: 75Q7413G"
1,2026-03-01 04:30:00,0.538709,"Clerking Doctor\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness. Worsening SOB over wks. Now limits daily acti vity. No CP. No syncope. No palps. Worse on exertion, unchanged at rest. No fever, cough, or sputum. No recent travel or surgery.\n\nReview of Systems\nCVS: No CP, no dizziness. Res: No haemoptysis, no wheeze. GI: No N&V, no diarrhoea. GU: No dysuriz, no frequency. CNS: No focal neuro deficits.\n\nPast Medical History\n- HTN\n- Asthma\n\nMedications\nNonereported\n\nAllergies\nNuts\n\nSocial History\n- Lives alone, independent in ADLs\n- Non-smooker\n- Rare alcohol use\n- No recreational drug use\n- No recent travel\n\nFamily History\nNo known FHx of thromboembolism or pulmonary HTN\n\nOn Examination\nPt appesrs SOB but alert. JVP raised to angle of jaw at 45Â°. HS: normal S1/S2, no murmurs, loud P2. Chest: clear bilaterally, no added sounds. No peripheral oedema. No clubbing.\n\nObservations\nHR 112\nBP 92/64\nRR 28\nTemp 36.8\nSpO2 990% on 2L NC\n\nInvestigations\nCXR: Cardiomegaly. Pending: CTPA, echo.\n\nTest Results\nNT-proBNP 4500 pg/mL. ABG: PaO2 8.5 kPa, PaCO2 3.5 kPq, pH 7.47.\n\nImpression\nChronic thromboembolic pulmonary HTN. Likely secondary tounprovoked PE.\n\nPlan\n1. Start enoxaparin 1.5 mg/kg daily (initial dose already given in ED). 2. CT pulmonary angiogram this morning to confirm diagnosis. 3. Echocardiogram today to assess right heart strain and pulmonary pressures. 4. Cardiology referral for further input on management odf pulmonary HTN. 5. Continue oxygen therapy to maintain SpO2 >92%. 6. Monitor clinical status closely, including repeat ABG and SpO2 mlonitoring as required.\n\n\nDr. Sade Olowoyeye (Specialty Registrar) \nGMC number: 1359799"
2,2026-03-01 11:00:00,0.531770,Patient assessed at bedside at 11:00 on 03/01/26 by Therapist Joan Winifred Shaw. Progressive breathlessness secondary to chronic thromboembolic pulmonary HTN noted. Diaphragmatic breathing exercises introduced to improve oxygenation and reduce dyspnoea. Technique explained and demonstrated. Patient instructde to perform exercises three times daily under supervision during hospital stay. Next steps: Daily physio sessions planned to enhance respiratory function and reduce exertional breathlessness.\nTherapist Joan Winifred Shaw (Physical)
3,2026-03-01 17:30:00,0.566744,"Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness\n\nIssues\n1. Chronic thromboembolic pulmonary hypertension (CTEPH)\n-CTPA confirmed chronic thromboembolivc disease\n-NT-proBNP significantly elevated (4,500 pg/mL)\n-Awaiting echo results for RV function\n\nToday\n\nOn Review\nReports no new symptoms. Comfortable on low-flow O2.\n\nInvestigations\nCYPA confirmed chronic thromboembolic disease. Awaiting echo results for RV function.\n\nObservations\nHR: 96\nBP: 118/72\nRR: 20\nTemp: 36.8\nSpO2: 94% on 2L O2\n\nOn Examination\nAlert, NAD. JVP raised. Heart sounds normal, no murmurs. Bibasal creps. No peripheral oedema.\n\nPlan\n1. Await echo results to assess RV function and determine se verity of pulm HTN. \n2. Note: Anticoagulation with apixaban 10 mg BD already initiated earlier today. Plan to optim

In [273]:
# Convert the retrieved BGE chunks into the dataframe format
# expected by generate_summary().
bge_whole_generation_df = prepare_for_generation(
    bge_whole_top20_chrono,
    SELECTED_PERSON_ID
)

# Generate the longitudinal summary using the SAME
# model, prompt, and temperature as the other configurations.
bge_whole_summary = generate_summary(
    patient_id=SELECTED_PERSON_ID,
    notes_df=bge_whole_generation_df,
    prompt=SYSTEM_PROMPT
)

print(bge_whole_summary)

**Longitudinal Clinical Summary – 03 Jan 2026 to Discharge (early Jan 2026)**  

**03 Jan 2026** – The patient (history of hypertension and asthma) presented with progressive breathlessness limiting daily activity. Initial assessment showed tachycardia (HR 112), hypotension (BP 92/64 mmHg), tachypnoea (RR 28), SpO₂ 89 % on room air (improved to 92 % with low‑flow O₂). Physical exam revealed raised JVP, loud P₂ and clear lungs. ABG demonstrated mild hypoxia (PaO₂ 8.5 kPa) with compensated respiratory alkalosis. NT‑proBNP was markedly elevated (3 200 pg/mL, later 4 500 pg/mL). CXR showed cardiomegaly.  

*Investigations*: CTPA (performed later) and transthoracic echocardiogram were ordered.  

*Management*: Enoxaparin 1.5 mg/kg daily was started (initial dose given in ED). Low‑flow O₂ was continued to keep SpO₂ > 92 %.  

**03 Jan 2026 (11:00)** – Physiotherapy introduced diaphragmatic breathing exercises (three times daily) and planned daily sessions.  

**03 Jan 2026 (ward round)** – C

In [275]:
# MedCPT uses a separate encoder for retrieval queries.
query_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
)

query_model = AutoModel.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
)

query_model.eval()

print("MedCPT Query Encoder loaded.")

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT Query Encoder loaded.


In [276]:
def embed_medcpt_query(text):
    # Convert the retrieval query into tokens understood by MedCPT.
    inputs = query_tokenizer(
        [text],
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # We only need embeddings, so gradients are unnecessary.
    with torch.no_grad():
        outputs = query_model(**inputs)

    # MedCPT uses the representation of the first token
    # as the query embedding.
    embedding = outputs.last_hidden_state[:, 0, :]

    return embedding.cpu().numpy()

In [282]:
def embed_medcpt_articles(texts, batch_size=8):
    """
    Embed clinical chunks using the MedCPT Article Encoder.

    MedCPT supports a maximum sequence length of 512 tokens,
    so longer whole-note chunks are explicitly truncated to 512.
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = article_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512   # Important for MedCPT
        )

        with torch.no_grad():
            outputs = article_model(**inputs)

        # Use first-token representation as the chunk embedding.
        batch_embeddings = outputs.last_hidden_state[:, 0, :]

        all_embeddings.append(batch_embeddings.cpu())

    return torch.cat(all_embeddings, dim=0).numpy()

In [283]:
# Query uses the MedCPT Query Encoder.
medcpt_query_embedding = embed_medcpt_query(
    RETRIEVAL_QUERY
)

# Clinical notes use the MedCPT Article Encoder.
medcpt_whole_embeddings = embed_medcpt_articles(
    patient_whole_chunks["chunk_text"].tolist()
)

print("Query embedding shape:", medcpt_query_embedding.shape)
print("Whole-note embeddings shape:", medcpt_whole_embeddings.shape)

Query embedding shape: (1, 768)
Whole-note embeddings shape: (45, 768)


In [284]:
# Calculate similarity between the MedCPT query embedding
# and each of the 45 whole-note embeddings.
medcpt_whole_scores = cosine_similarity(
    medcpt_query_embedding,
    medcpt_whole_embeddings
)[0]

# Attach the similarity score to each whole-note chunk.
medcpt_whole_results = patient_whole_chunks.copy()
medcpt_whole_results["similarity_score"] = medcpt_whole_scores

# Rank all 45 chunks from most relevant to least relevant.
medcpt_whole_ranked = (
    medcpt_whole_results
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

# Use our frozen retrieval depth.
medcpt_whole_top20 = medcpt_whole_ranked.head(20).copy()

print("Total ranked chunks:", len(medcpt_whole_ranked))
print("Retrieved chunks:", len(medcpt_whole_top20))

medcpt_whole_top20[
    ["creation_timestamp", "similarity_score", "chunk_text"]
]

Total ranked chunks: 45
Retrieved chunks: 20


,creation_timestamp,similarity_score,chunk_text
0,2026-03-01 04:00:00,0.648643,"Patient\nAbena Bonsu\n\nAge\n43\n\nSex\nFemale\n\nNHS No.:\n395107142\n\nDate/Time\n03/01/2604:00\n\nSeen By\nDr. Brenda Veronica Miles, ED Consultant\n\nPresenting Complaint\nProgressive breathlessness\n\nHistory of Presenting Complaint\n- Progressive worsening of SOB over the last 2 weks\n- SOB worse on exertion\n- No associated CP, haemoptysis, fever, or syncope reported\n- Describes a sensation of tightness in the chest but denies palpitations or wheezing\n\nPast Medical History\n- HTN\n- Asthma\n\nMedications\n- None reported by the patient at the time of admjssion\n\nAllergies\n- Nuts\n\nSocial History\n- Lives alone\n- Non-smoker\n- No alcohol consumption reported\n- Works as a school teacher\n\nFamily History\n- Father: HTN\n- Mother: Asthma\n\nSystems Review\n- Respiratory: Progressive exertional dyspnoea. No haemoptysis, wheezing, or significant cough.\n- Cardiovascular: No palpitations, chest pain, or syncope.\n- Gastrointestinal: No abdominal pain, nausea, or vomitimg.\n- Neurological: No dizziness, confusion, or focal neuro symptoms.\n- General: No fever or weight loss.\n\nOn Examination\n- Appears mildly distressed with laboured breathing\n- Cardiovascular: Elevated JVP. No peripheral oedema noted. Heart sounds normal, no murmurs.\n- Respiratory: Bilateral basal creps heard on aauscultation. No wheeze.\n- Abdomen: Mildly tenddr hepatomegaly. No ascites.\n- Peripheral vascular: No evidence of DVT. No calf tenderness or swelling.\n\nObservations\nBP 92/64 mmHg\nHR 112 bpm\nRR 28 breaths/min\nTemp 36.8Â°C\nSPO2 90% on 2L NC O2\n\nInvestigations\n- Chest X-ray: Bilateral pulmonary congestion, no focal consolidation or pneumothorax. - CT pulmonary angiogram and echocardiogram planed to confirm suspected chronic thromboembolic pulmonary HTN (pending results).\n\nTest Results\n- ABG: PaO2 8.5 kPa, PaCO2 3.8 kPa, pH 7.46, HCO3- 24 mmol/L (compensated rdspiratory alkalosis, mild hypoxia)\n- NT-proBNP: 4,500 pg/mL (elevated, subsequent result after earlier 3,200 pg/mL reading)\n- Blood tests: D-dimer elevated, other results pending.\n\nImpression\nSuspected chronic thromboembolic pulmonary HTN causing progressive breathlessness and cardiac strain.\n\nReferral\nAccepted by Respiratory HDU under Dr. Elizabeth Kathryn Singh. Prepared for transfer to bed location B01 with asdistance by Nurse Sarabjit Gupta.\n\nPlan\n- Start therapeutic-dose enoxaparin 1.5 mg/kg daily subcutaneously for anticoagulation.\n- Arrange CT pulmonary angiogram to confirm diagnosis of chronic thromboembolic pulmonary HTN.\n- Perform echocardiogram to assess right heart strain and pulmonary presures.\n- Admit to Respiratory HDU for close monitoring and specialist input.\n- Continue oxygen therapy to address hypoxia and respiratory distress.\n\n\nDr. Brenda Veronica Miles (ED Consultant) \nGMC number: 1432794"
1,2026-05-01 16:00:00,0.645578,"Patient attended an education session led by me. Discussed pulmonary HTN, anticoagulation therapy, and prevention of complications, including blood clots. No medication changes made. Patient encouraged to ask further questions during reviews.\nDr. Anne Marion Perkins \nGMC number: 1866682"
2,2026-10-01 09:30:00,0.643867,- Provided patient with detailed discharge summary.\n- Explained outpatient follow-up plan.\n- Shared contact details for local pulmonary HTN support services.\n- Advised no new medications or treatment cha nges.\n- Patient instructed to contact support services if concerns or assistance needed post-discharge.\nNurse Audrey Phyllis Gill \nNMC number: 75Q7413G
3,2026-05-01 08:00:00,0.641501,"Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness\n\nIssues\n1. Chronic Thromboembolic Pulmonary Hypertension (CTEPH)\n-Persistent NT-proBNP elevation (3200 pg/mL)\n-Ongoing oxygen dependency (2L O2)\n-Clinically stable with reduced oxygen requirements\n\nToday\n\nOn Review\nFeels

In [285]:
# Keep the exact 20 chunks MedCPT retrieved,
# but put them back into clinical chronological order.
medcpt_whole_top20_chrono = (
    medcpt_whole_top20
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Retrieved chunks:", len(medcpt_whole_top20_chrono))

medcpt_whole_top20_chrono[
    ["creation_timestamp", "similarity_score", "chunk_text"]
]

Retrieved chunks: 20


,creation_timestamp,similarity_score,chunk_text
0,2026-03-01 03:15:00,0.616747,"- Date: 03/01/26\n - Time: 03:15\n - Staff: Nurse Audrey Phyllis Gill\n - Chief complaint: Progressive breathlessness.\n - Past medical history: HTN, Asthma.\n - Initial management: Patient placed on low-flow oxygen therapy to improve oxygenation, with SpO2 improving from 89%to 92%.\n - ABG results reviewed: PaO2 8.5 kPa, indicating mild hypoxia; compensated respiratory alkalosis likely due to hyperventilation.\n - NT-proBNP elevated at 3,200 pg/mL, suggesting significant cardiac strain.\n - Findings escalated to the medical team for further evaluation and consideration of imaging to investigate suspected pulmonary HTN.\nNurse Audrey Phyllis Gill \nNMC number: 75Q7413G"
1,2026-03-01 04:00:00,0.648643,"Patient\nAbena Bonsu\n\nAge\n43\n\nSex\nFemale\n\nNHS No.:\n395107142\n\nDate/Time\n03/01/2604:00\n\nSeen By\nDr. Brenda Veronica Miles, ED Consultant\n\nPresenting Complaint\nProgressive breathlessness\n\nHistory of Presenting Complaint\n- Progressive worsening of SOB over the last 2 weks\n- SOB worse on exertion\n- No associated CP, haemoptysis, fever, or syncope reported\n- Describes a sensation of tightness in the chest but denies palpitations or wheezing\n\nPast Medical History\n- HTN\n- Asthma\n\nMedications\n- None reported by the patient at the time of admjssion\n\nAllergies\n- Nuts\n\nSocial History\n- Lives alone\n- Non-smoker\n- No alcohol consumption reported\n- Works as a school teacher\n\nFamily History\n- Father: HTN\n- Mother: Asthma\n\nSystems Review\n- Respiratory: Progressive exertional dyspnoea. No haemoptysis, wheezing, or significant cough.\n- Cardiovascular: No palpitations, chest pain, or syncope.\n- Gastrointestinal: No abdominal pain, nausea, or vomitimg.\n- Neurological: No dizziness, confusion, or focal neuro symptoms.\n- General: No fever or weight loss.\n\nOn Examination\n- Appears mildly distressed with laboured breathing\n- Cardiovascular: Elevated JVP. No peripheral oedema noted. Heart sounds normal, no murmurs.\n- Respiratory: Bilateral basal creps heard on aauscultation. No wheeze.\n- Abdomen: Mildly tenddr hepatomegaly. No ascites.\n- Peripheral vascular: No evidence of DVT. No calf tenderness or swelling.\n\nObservations\nBP 92/64 mmHg\nHR 112 bpm\nRR 28 breaths/min\nTemp 36.8Â°C\nSPO2 90% on 2L NC O2\n\nInvestigations\n- Chest X-ray: Bilateral pulmonary congestion, no focal consolidation or pneumothorax. - CT pulmonary angiogram and echocardiogram planed to confirm suspected chronic thromboembolic pulmonary HTN (pending results).\n\nTest Results\n- ABG: PaO2 8.5 kPa, PaCO2 3.8 kPa, pH 7.46, HCO3- 24 mmol/L (compensated rdspiratory alkalosis, mild hypoxia)\n- NT-proBNP: 4,500 pg/mL (elevated, subsequent result after earlier 3,200 pg/mL reading)\n- Blood tests: D-dimer elevated, other results pending.\n\nImpression\nSuspected chronic thromboembolic pulmonary HTN causing progressive breathlessness and cardiac strain.\n\nReferral\nAccepted by Respiratory HDU under Dr. Elizabeth Kathryn Singh. Prepared for transfer to bed location B01 with asdistance by Nurse Sarabjit Gupta.\n\nPlan\n- Start therapeutic-dose enoxaparin 1.5 mg/kg daily subcutaneously for anticoagulation.\n- Arrange CT pulmonary angiogram to confirm diagnosis of chronic thromboembolic pulmonary HTN.\n- Perform echocardiogram to assess right heart strain and pulmonary presures.\n- Admit to Respiratory HDU for close monitoring and specialist input.\n- Continue oxygen therapy to address hypoxia and respiratory distress.\n\n\nDr. Brenda Veronica Miles (ED Consultant) \nGMC number: 1432794"
2,2026-03-01 04:30:00,0.622630,"Clerking Doctor\nDr. Sade Olowoyeye (SpR)\n\nPresenting Complaint\nProgressive breathlessness. Worsening SOB over wks. Now limits daily acti vity. No CP. No syncope. No palps. Worse on exertion, unchanged at rest. No fever, cough, or sputum. No recent travel or surgery.\n\nReview of Systems\nCVS: No CP, no dizziness. Res: No haemoptysis, no 

In [286]:
# Convert the retrieved chunks into the dataframe format
# expected by our existing generate_summary() function.
medcpt_whole_generation_df = prepare_for_generation(
    medcpt_whole_top20_chrono,
    SELECTED_PERSON_ID
)

# Generate the longitudinal summary using the same
# GPT-OSS-120B model, SYSTEM_PROMPT, and temperature=0.
medcpt_whole_summary = generate_summary(
    SELECTED_PERSON_ID,
    medcpt_whole_generation_df,
    SYSTEM_PROMPT
)

print(medcpt_whole_summary)

**Patient:** Abena Bonsu, 43‑year‑old female  
**Key past history:** Hypertension, asthma; nut allergy  

**03 Jan 2026 – Emergency Department**  
- Presentation: 2‑week progressive exertional dyspnoea, no chest pain, cough, wheeze, fever or syncope.  
- Vitals: BP 92/64 mmHg, HR 112 bpm, RR 28 /min, SpO₂ 90 % on 2 L nasal cannula, Temp 36.8 °C.  
- Examination: mild distress, raised JVP, bibasal crepitations, mild hepatomegaly, no peripheral oedema.  
- Initial labs: ABG – PaO₂ 8.5 kPa, PaCO₂ 3.8 kPa, pH 7.46 (compensated respiratory alkalosis, mild hypoxia); NT‑proBNP 3 200 pg/mL (later 4 500 pg/mL); D‑dimer elevated.  
- Imaging: CXR – bilateral pulmonary congestion, cardiomegaly; CT pulmonary angiogram (CTPA) and transthoracic echocardiogram ordered.  
- Impression: suspected chronic thromboembolic pulmonary hypertension (CTEPH) with cardiac strain.  
- Initial management: low‑flow oxygen, therapeutic enoxaparin 1.5 mg/kg SC daily, admission to Respiratory HDU.

**Early inpatient c